<img src="https://img.shields.io/badge/Grafos-Aula%20pr%C3%A1tica-1f4e79?style=for-the-badge" alt="badge"> <img src="https://img.shields.io/badge/NetworkX-3.x-orange?style=for-the-badge" alt="badge"> <img src="https://img.shields.io/badge/spaCy-NER%20como%20ferramenta-09a3d5?style=for-the-badge" alt="badge"> <img src="https://img.shields.io/badge/Plotly-interativo-9085e9?style=for-the-badge" alt="badge">

# 🕸️ Do texto à rede · co-ocorrência de entidades nomeadas

**Aula prática** · Algoritmos e Estrutura de Dados II (Grafos) · UFRN / DCA · ivanovitch.silva@ufrn.br

> Em 1993, Donald Knuth leu *Os Miseráveis* com lápis na mão e anotou, capítulo a capítulo, quais personagens apareciam juntos. O resultado é a rede que hoje vem embutida no NetworkX (`nx.les_miserables_graph()`). Nesta aula fazemos o trabalho de Knuth em segundos, em português, com um romance que a turma inteira já ouviu falar. A pergunta que o Brasil discute há mais de um século (Capitu traiu ou não traiu?) uma rede não responde. Mas ela responde outra, mais modesta e mensurável.
> **Quem aparece com quem, quantas vezes, e que estrutura isso forma?**

Até aqui construímos redes de três jeitos (arquivo CSV, à mão, coleta na Internet). Hoje entra a quarta fonte, a mais abundante de todas, **texto**. Um texto não tem nós nem arestas explícitos. Ele tem palavras. A ponte entre palavras e grafo é uma ideia de uma linha, e é ela que organiza o notebook inteiro.

> **Duas coisas estão na mesma unidade de texto ⟹ existe uma aresta entre elas.**

Para transformar essa frase em código é preciso tomar três decisões, e cada uma muda a rede resultante.

| Verbo | Pergunta | Decisão de modelagem |
|---|---|---|
| **fatiar** | o que é uma *unidade*? | capítulo, parágrafo, sentença, documento, janela de $w$ sentenças |
| **extrair** | o que é um *item*? | pessoas, lugares, organizações, palavras-chave, *hashtags* |
| **ligar** | o que é *estar junto*? | contagem, presença, Jaccard, PMI |

Para o verbo **extrair** usaremos uma ferramenta emprestada do processamento de linguagem natural, o **reconhecimento de entidades nomeadas** (NER, de *named entity recognition*). Ela ocupa uma seção curta e uma única função. O resto do notebook é grafo.

## 🎯 O que você vai conseguir fazer ao final

1. Definir rede de co-ocorrência como **projeção de um grafo bipartido** (unidades × itens) e calcular pesos por contagem e por Jaccard.
2. Montar a matriz de incidência $B$ e obter a co-ocorrência por $B^{\top}B$, ligando isso ao que já sabe sobre matrizes de adjacência e passeios.
3. Usar NER como **ferramenta com contrato**, uma função que recebe texto e devolve uma tabela, sem precisar abrir o modelo.
4. Transformar **menções** em **nós** (normalização, dicionário de aliases, contração) e explicar por que projeção e contração não comutam.
5. Escolher a **janela** de co-ocorrência e prever o efeito sobre densidade, clustering e componentes.
6. Ler a rede com o arsenal das aulas anteriores (grau e força, intermediação, k-core, componentes) e extrair o **esqueleto** por limiar de peso.
7. Reconhecer o **artefato do clique** nas projeções e corrigi-lo com normalização.

## 🗺️ Mapa do notebook

| Seção | Pergunta que responde | Dados |
|---|---|---|
| 1 | Como preparar o ambiente e o estúdio visual? | · |
| 2 | O que é, formalmente, uma rede de co-ocorrência? | nenhum |
| 3 | Como o mecanismo funciona, passo a passo, em um texto de seis frases? | texto de bolso |
| 4 | 🧰 Como usar NER sem precisar entender NER? | texto de bolso |
| 5 | Quem aparece com quem em *Dom Casmurro*? | romance, 148 capítulos |
| 6 | O que muda quando as unidades são documentos e os itens têm tipos? | *A Semana*, 244 crônicas |
| 7 | Desafios (três resolvidos, três abertos) | os dois corpora |
| 8 | Quiz, checklist e referências | · |

## 🧠 Por que Gestalt?

Os princípios de Gestalt descrevem como o cérebro agrupa o que vê. Nas aulas anteriores usamos isso nas figuras. Aqui usamos também na **arquitetura do notebook**, porque a ordem em que as ideias chegam muda o que fica na memória.

| Princípio | No desenho do notebook | Nas figuras |
|---|---|---|
| **Todo antes das partes** | a rede final aparece antes do mecanismo; o mecanismo aparece em seis frases antes de aparecer em um romance | · |
| **Figura e fundo** | o grafo é a figura; o NER é o fundo, confinado às caixas 🧰 | o que está em discussão fica colorido, o resto fica cinza |
| **Proximidade** | a explicação fica colada ao código que explica; a pergunta fica colada à saída que a responde | nós que co-ocorrem mais ficam mais perto (*layout* ponderado) |
| **Similaridade** | as caixas recorrentes têm sempre o mesmo formato (⏸️ pare e pense, 🧠 Gestalt em ação, 🧰 ferramenta, 🧭 de volta ao grafo, ✅ o que fica) | mesma cor para o mesmo tipo de entidade em todas as figuras; magnitude sempre em uma única matiz azul |
| **Continuidade** | uma única linha narrativa, texto de bolso → romance → crônicas → desafios | uma aresta da projeção é um caminho de comprimento 2 no grafo bipartido |
| **Fechamento** | cada seção termina em ✅ *O que fica*; o checklist final fecha os objetivos do início | limiares de peso "fecham" o esqueleto da rede |
| **Destino comum** | · | animações em que arestas somem juntas quando o limiar sobe, ou nós acendem juntos quando o ano muda |
| **Experiência passada** | mesma paleta e mesmas funções de desenho das aulas de Hubs e de k-core | · |

# 1 · Preparação do ambiente

Três dependências além do que já usamos. O **spaCy** com o modelo em português `pt_core_news_md` (a ferramenta NER, cerca de 40 MB), o **Plotly** para figuras interativas e o corpus **machado** do NLTK, que traz a obra completa de Machado de Assis em texto puro (domínio público). No Colab a célula abaixo instala o que faltar; leva em torno de um minuto na primeira execução.

In [1]:
import importlib, subprocess, sys

def garantir(modulo, pacote=None):
    if importlib.util.find_spec(modulo) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pacote or modulo], check=False)
        importlib.invalidate_caches()

for modulo, pacote in [("spacy", "spacy"), ("plotly", "plotly"), ("nltk", "nltk")]:
    garantir(modulo, pacote)

import spacy
MODELO = "pt_core_news_md"
if importlib.util.find_spec(MODELO) is None:
    subprocess.run([sys.executable, "-m", "spacy", "download", MODELO, "-q"], check=False)
    importlib.invalidate_caches()
try:
    nlp = spacy.load(MODELO)
except OSError:                       # recém-instalado na mesma sessão: carrega pelo pacote
    nlp = importlib.import_module(MODELO).load()
print("spaCy", spacy.__version__, "| modelo", MODELO, "| componentes:", nlp.pipe_names)

spaCy 3.8.16 | modelo pt_core_news_md | componentes: ['tok2vec', 'morphologizer', 'parser', 'lemmatizer', 'attribute_ruler', 'ner']


In [2]:
import os, re, io, math, time, zipfile, itertools, collections, textwrap
import numpy as np
import pandas as pd
import networkx as nx
from networkx.algorithms import bipartite
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

print("NetworkX", nx.__version__, "| pandas", pd.__version__)

NetworkX 3.6.1 | pandas 2.2.3


## 1.1 · O corpus machado

O NLTK distribui a obra de Machado de Assis (romances, contos, crônicas, poesia) em um único arquivo `machado.zip`, codificado em `cp1252`. Vamos baixá-lo uma vez e ler os arquivos direto do zip, sem passar pelo leitor de corpus do NLTK (que exigiria outros downloads). Se o download pelo NLTK falhar, a célula tenta a URL original do arquivo.

In [3]:
import nltk

def localizar_zip():
    for base in nltk.data.path + [os.path.expanduser("~/nltk_data")]:
        caminho = os.path.join(base, "corpora", "machado.zip")
        if os.path.exists(caminho):
            return caminho
    return None

ZIP_MACHADO = localizar_zip()
if ZIP_MACHADO is None:
    nltk.download("machado", quiet=True)
    ZIP_MACHADO = localizar_zip()
if ZIP_MACHADO is None:                      # plano B, baixar direto do repositório do NLTK
    import requests
    url = "https://raw.githubusercontent.com/nltk/nltk_data/gh-pages/packages/corpora/machado.zip"
    destino = os.path.join(os.path.expanduser("~/nltk_data"), "corpora")
    os.makedirs(destino, exist_ok=True)
    with open(os.path.join(destino, "machado.zip"), "wb") as f:
        f.write(requests.get(url, timeout=120).content)
    ZIP_MACHADO = localizar_zip()

def ler_machado(nome):
    # nome no formato 'romance/marm08.txt'; o corpus está em cp1252 (travessões e aspas curvas incluídos)
    with zipfile.ZipFile(ZIP_MACHADO) as z:
        return z.read(f"machado/{nome}").decode("cp1252", errors="replace")

with zipfile.ZipFile(ZIP_MACHADO) as z:
    indice = z.read("machado/CONTENTS").decode("cp1252", errors="replace")
print(ZIP_MACHADO)
print(indice[:900])

/root/nltk_data/corpora/machado.zip
Machado de Assis -- Obra Completa

Romance

romance/marm01.txt: Ressurreição (1872)
romance/marm02.txt: A Mão e a Luva (1874)
romance/marm03.txt: Helena (1876)
romance/marm04.txt: Iaiá Garcia (1878)
romance/marm05.txt: Memórias Póstumas de Brás Cubas (1881)
romance/marm06.txt: Casa Velha (1885)
romance/marm07.txt: Quincas Borba (1891)
romance/marm08.txt: Dom Casmurro (1899)
romance/marm09.txt: Esaú e Jacó (1904)
romance/marm10.txt: Memorial de Aires (1908)

Poesia

poesia/maps01.txt: Crisálidas (1864)
poesia/maps02.txt: Falenas (1870)
poesia/maps03.txt: Americanas (1875)
poesia/maps04.txt: Gazeta de Holanda (1886-88)
poesia/maps05.txt: Ocidentais (1901)
poesia/maps06.txt: O Almada (1908)
poesia/maps07.txt: Dispersas (1854-1939)

Contos

contos/macn001.txt: Contos Fluminenses (1870); Miss Dollar; Luís Soares; A mulher de preto; O segredo de Augusta; Confissões de uma viúva moça; Linha ret


## 1.2 · Paleta e estúdio visual

As mesmas funções das aulas de Hubs e de k-core, mais duas coisas novas. Uma cor **fixa por tipo de entidade** (pessoa, lugar, organização, outro), usada em todas as figuras, e um traçado de arestas em **classes de largura**, porque em uma rede de co-ocorrência o peso da aresta é o dado principal. Como antes, magnitude usa uma única matiz azul e o que não é o assunto da figura fica cinza.

In [4]:
CAT = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#9085e9", "#e34948"]
AZUIS = ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#1c5cab", "#104281", "#0d366b"]
CINZA_NO, CINZA_ARESTA, CINZA_TEXTO = "#c3c2b7", "#d9d8d2", "#52514e"
DESTAQUE, SUPERFICIE, GRADE = "#e34948", "#fcfcfb", "#e1e0d9"

# cor por tipo de entidade: a mesma em todas as figuras do notebook (similaridade)
TIPO_COR = {"PER": CAT[0], "LOC": CAT[1], "ORG": CAT[2], "MISC": CAT[6]}
TIPO_NOME = {"PER": "pessoa", "LOC": "lugar", "ORG": "organização", "MISC": "outro"}

def rampa_azul(valores, vmin=None, vmax=None):
    v = np.asarray(list(valores), dtype=float)
    vmin = v.min() if vmin is None else vmin
    vmax = v.max() if vmax is None else vmax
    if vmax == vmin:
        return [AZUIS[-1]] * len(v)
    idx = np.round((v - vmin) / (vmax - vmin) * (len(AZUIS) - 1)).astype(int)
    return [AZUIS[i] for i in idx]

def traco_arestas(g, pos, cor=CINZA_ARESTA, largura=1.0, opacidade=0.7, nome="arestas", arestas=None):
    xs, ys = [], []
    for u, v in (arestas if arestas is not None else g.edges()):
        xs += [pos[u][0], pos[v][0], None]
        ys += [pos[u][1], pos[v][1], None]
    return go.Scatter(x=xs, y=ys, mode="lines", hoverinfo="skip", name=nome,
                      line=dict(color=cor, width=largura), opacity=opacidade, showlegend=False)

def cortes_de_peso(g, peso="weight"):
    # três cortes (quantis 50, 80 e 95 %) que separam quatro classes de largura de aresta
    pesos = np.array([d.get(peso, 1) for _, _, d in g.edges(data=True)], dtype=float)
    return np.quantile(pesos, [0.5, 0.8, 0.95]) if len(pesos) else np.array([1.0, 2.0, 3.0])

def tracos_arestas_ponderadas(g, pos, cor="#8f9bb3", classes=(1, 2, 4, 7), peso="weight", opacidade=0.55, cortes=None):
    # divide as arestas em quatro classes de largura; devolve SEMPRE quatro traces (vazias, se preciso),
    # para que animações troquem quadro a quadro sem deixar arestas "fantasma" do quadro anterior
    cortes = cortes_de_peso(g, peso) if cortes is None else cortes
    tracos = []
    for k, largura in enumerate(classes):
        if k == 0:
            sel = [(u, v) for (u, v, d) in g.edges(data=True) if d.get(peso, 1) <= cortes[0]]
        elif k < 3:
            sel = [(u, v) for (u, v, d) in g.edges(data=True) if cortes[k - 1] < d.get(peso, 1) <= cortes[k]]
        else:
            sel = [(u, v) for (u, v, d) in g.edges(data=True) if d.get(peso, 1) > cortes[2]]
        tracos.append(traco_arestas(g, pos, cor=cor, largura=largura, opacidade=opacidade, arestas=sel))
    return tracos

def traco_nos(g, pos, cores, tamanhos=14, hover=None, rotulos=True, nome="nós", opacidade=1.0, borda="#ffffff",
              posicao_texto="middle center", tamanho_fonte=11):
    nos = list(g.nodes())
    cores_lista = cores if isinstance(cores, list) else [cores] * len(nos)
    claras = set(AZUIS[:3]) | {CINZA_NO, CAT[3]}
    cor_texto = [("#0b0b0b" if (c in claras or posicao_texto != "middle center") else "white") for c in cores_lista]
    return go.Scatter(
        x=[pos[n][0] for n in nos], y=[pos[n][1] for n in nos],
        mode="markers+text" if rotulos else "markers",
        text=[str(n) for n in nos] if rotulos else None, textposition=posicao_texto,
        textfont=dict(color=cor_texto, size=tamanho_fonte),
        hovertext=hover if hover is not None else [str(n) for n in nos], hoverinfo="text",
        marker=dict(color=cores, size=tamanhos, line=dict(color=borda, width=1.5), opacity=opacidade),
        name=nome, showlegend=False)

def figura_rede(tracos, titulo="", altura=520, largura=None):
    fig = go.Figure(data=tracos)
    fig.update_layout(
        title=dict(text=titulo, x=0.02, font=dict(size=17, color="#0b0b0b")),
        height=altura, width=largura, plot_bgcolor=SUPERFICIE, paper_bgcolor=SUPERFICIE,
        margin=dict(l=10, r=10, t=50, b=10), hovermode="closest",
        xaxis=dict(visible=False), yaxis=dict(visible=False))
    return fig

def tamanho_por(valores, minimo=9, maximo=40):
    v = np.asarray(list(valores), dtype=float)
    if v.max() == v.min():
        return [float(minimo + maximo) / 2] * len(v)
    return list(minimo + (maximo - minimo) * np.sqrt((v - v.min()) / (v.max() - v.min())))

def legenda_tipos(fig, tipos_presentes=("PER", "LOC", "ORG", "MISC")):
    # marcadores fantasmas só para a legenda de tipos
    for t in tipos_presentes:
        fig.add_scatter(x=[None], y=[None], mode="markers", marker=dict(color=TIPO_COR[t], size=12),
                        name=TIPO_NOME[t], showlegend=True)
    fig.update_layout(legend=dict(orientation="h", y=-0.02, x=0, title_text=""))
    return fig

print("estúdio pronto")

estúdio pronto


## 1.3 · O todo antes das partes

Antes de qualquer definição, a rede que motivou a aula. É a rede de *Os Miseráveis* que Knuth montou à mão e que o NetworkX distribui. Cada nó é um personagem, cada aresta liga dois personagens que aparecem no mesmo capítulo, e o peso é o número de capítulos compartilhados. No fim do notebook teremos construído uma rede exatamente como esta, para um romance brasileiro, sem lápis.

> 🧠 **Gestalt em ação · Proximidade + Figura e fundo.** O *layout* usa os pesos, então quem coaparece muito fica perto (Valjean, Cosette, Marius, Javert no centro). O tamanho do nó é a força (soma dos pesos). Passe o mouse para ler os nomes. Não é preciso conhecer o livro para ver o enredo na forma da rede, um miolo denso e vários grupos pendurados nele.

In [5]:
les_mis = nx.les_miserables_graph()
nx.set_node_attributes(les_mis, dict(les_mis.degree(weight="weight")), "unidades")   # força, só para o tamanho
pos_lm = nx.spring_layout(les_mis, seed=3, weight="weight", k=0.35)
nos_lm = list(les_mis.nodes())
forca_lm = dict(les_mis.degree(weight="weight"))
fig = figura_rede(tracos_arestas_ponderadas(les_mis, pos_lm)
                  + [traco_nos(les_mis, pos_lm, AZUIS[5], tamanho_por(forca_lm.values(), 6, 34),
                               [f"<b>{n}</b><br>grau {les_mis.degree(n)} · força {forca_lm[n]}" for n in nos_lm], rotulos=False)],
                  f"Os Miseráveis (Knuth, 1993) · {les_mis.number_of_nodes()} personagens · largura = capítulos compartilhados", altura=560)
fig.show()

# 2 · O que é uma rede de co-ocorrência

## 2.1 · Três conjuntos e uma projeção

Seja $U$ um conjunto de **unidades de texto** (capítulos, parágrafos, sentenças, documentos) e $I$ um conjunto de **itens** (para nós, entidades nomeadas). A relação "o item $i$ aparece na unidade $u$" define um **grafo bipartido** $\mathcal{B} = (U \cup I, E)$, com uma aresta $(u, i)$ para cada ocorrência distinta.

A **rede de co-ocorrência** é a **projeção** de $\mathcal{B}$ sobre $I$. Dois itens são vizinhos se compartilham ao menos uma unidade, e o **peso** da aresta é o número de unidades compartilhadas.

$$w(i, j) = \big|\, N(i) \cap N(j) \,\big|, \qquad N(i) = \{\, u \in U : (u, i) \in E \,\}$$

Isso é exatamente a projeção ponderada que vimos na aula de Elementos de Redes II. Lá projetamos pessoas × eventos; aqui projetamos unidades × entidades. **Mesma operação, outro domínio.**

## 2.2 · A mesma coisa em matrizes

Escreva a **matriz de incidência** $B$, com $|U|$ linhas e $|I|$ colunas, $B_{ui} = 1$ se o item $i$ aparece na unidade $u$. Então

$$C = B^{\top} B, \qquad C_{ij} = \sum_{u} B_{ui} B_{uj}$$

conta, para cada par $(i, j)$, em quantas unidades os dois aparecem juntos. A diagonal $C_{ii}$ é o número de unidades em que $i$ aparece (a "frequência documental" do item). Fora da diagonal, $C$ é a matriz de adjacência ponderada da rede de co-ocorrência.

Repare na conexão com a aula de passeios. No grafo bipartido inteiro, a matriz de adjacência é $A = \begin{pmatrix} 0 & B \\ B^{\top} & 0 \end{pmatrix}$ e o bloco inferior direito de $A^{2}$ é $B^{\top}B$. Uma aresta da projeção é um **passeio de comprimento 2** no bipartido, item → unidade → item. Contar co-ocorrências é contar passeios de comprimento 2.

## 2.3 · O artefato do clique

Uma unidade com $k$ itens gera $\binom{k}{2} = k(k-1)/2$ arestas de uma vez. Um capítulo com 20 personagens produz 190 arestas; um documento com 50 entidades produz 1.225. Isso tem duas consequências que reaparecem o notebook inteiro.

* **Custo.** Construir a projeção custa $O\!\left(\sum_{u} k_u^{2}\right)$, dominado pelas unidades grandes.
* **Estrutura.** Uma única unidade grande vira um clique na projeção e distorce densidade, clustering e k-core. Quem viu a aula de k-core lembra do problema dos artigos com centenas de autores. É o mesmo fenômeno.

O remédio é **normalizar** o peso pela frequência dos dois itens. O índice de **Jaccard** é o mais simples.

$$J(i, j) = \frac{|N(i) \cap N(j)|}{|N(i) \cup N(j)|} = \frac{C_{ij}}{C_{ii} + C_{jj} - C_{ij}}$$

Jaccard vale 1 quando dois itens **só** aparecem juntos e tende a zero quando um deles aparece em toda parte. Contagem premia os hubs; Jaccard premia os pares exclusivos. As duas leituras são úteis e respondem perguntas diferentes.

## 2.4 · O que a co-ocorrência não é

Co-ocorrência é **presença conjunta**, nada mais. "Diego nunca esteve em Natal" liga Diego a Natal com a mesma força que "Diego mora em Natal". A rede não lê negações, não sabe quem falou com quem, não distingue amigo de inimigo. Ela mede uma coisa só, e mede bem, **atenção compartilhada** dentro de uma unidade de texto. Toda interpretação que fizermos precisa caber nesse limite.

# 3 · O mecanismo em uma tela

Antes de soltar o pipeline em um romance de 148 capítulos, vamos vê-lo inteiro em um texto de **seis frases**, com as entidades marcadas à mão. A ideia é que cada aresta da rede final possa ser conferida lendo o texto. Se algo não bater, o erro está no mecanismo, não nos dados.

In [6]:
TEXTO_BOLSO = [
    "Ana e Bruno estudam na UFRN.",
    "Bruno mora em Natal, perto da UFRN.",
    "Carla visitou Natal no feriado.",
    "Ana viajou para Recife com Carla.",
    "Em Recife, Carla conheceu Diego.",
    "Diego nunca esteve em Natal.",
]

# unidade = sentença; itens marcados à mão (por enquanto, sem nenhuma ferramenta)
ITENS_BOLSO = {
    1: ["Ana", "Bruno", "UFRN"],
    2: ["Bruno", "Natal", "UFRN"],
    3: ["Carla", "Natal"],
    4: ["Ana", "Recife", "Carla"],
    5: ["Recife", "Carla", "Diego"],
    6: ["Diego", "Natal"],
}
TIPO_BOLSO = {"Ana": "PER", "Bruno": "PER", "Carla": "PER", "Diego": "PER",
              "Natal": "LOC", "Recife": "LOC", "UFRN": "ORG"}

for u, frase in enumerate(TEXTO_BOLSO, start=1):
    print(f"S{u}  {frase:40s} -> {ITENS_BOLSO[u]}")

S1  Ana e Bruno estudam na UFRN.             -> ['Ana', 'Bruno', 'UFRN']
S2  Bruno mora em Natal, perto da UFRN.      -> ['Bruno', 'Natal', 'UFRN']
S3  Carla visitou Natal no feriado.          -> ['Carla', 'Natal']
S4  Ana viajou para Recife com Carla.        -> ['Ana', 'Recife', 'Carla']
S5  Em Recife, Carla conheceu Diego.         -> ['Recife', 'Carla', 'Diego']
S6  Diego nunca esteve em Natal.             -> ['Diego', 'Natal']


## 3.1 · Fatiar e extrair já estão feitos. Falta ligar

O bipartido é a tradução literal da tabela acima. Cada sentença é um nó de um lado, cada item é um nó do outro, cada ocorrência é uma aresta.

In [7]:
B_bolso = nx.Graph()
unidades = [f"S{u}" for u in ITENS_BOLSO]
itens = sorted({i for lista in ITENS_BOLSO.values() for i in lista})
B_bolso.add_nodes_from(unidades, bipartite=0)
B_bolso.add_nodes_from(itens, bipartite=1)
B_bolso.add_edges_from((f"S{u}", i) for u, lista in ITENS_BOLSO.items() for i in lista)

print(B_bolso)
print("é bipartido?", bipartite.is_bipartite(B_bolso))

Graph with 13 nodes and 16 edges
é bipartido? True


> 🧠 **Gestalt em ação · Continuidade.** À esquerda, o bipartido em duas colunas. À direita, a projeção. Passe o mouse em uma aresta da projeção e procure, à esquerda, o caminho de comprimento 2 que a justifica (item → sentença → item). A largura da aresta é o número desses caminhos.

In [8]:
P_bolso = bipartite.weighted_projected_graph(B_bolso, itens)

# duas colunas: sentenças à esquerda (em ordem), itens à direita (em ordem alfabética)
pos_b = {u: (0.0, -k) for k, u in enumerate(unidades)}
pos_b.update({i: (1.0, -k * (len(unidades) - 1) / (len(itens) - 1)) for k, i in enumerate(itens)})
pos_p = nx.circular_layout(P_bolso)

fig = make_subplots(rows=1, cols=2, subplot_titles=("grafo bipartido (sentenças × itens)", "projeção sobre os itens"),
                    horizontal_spacing=0.06)
# esquerda: rótulos ao lado dos nós
fig.add_trace(traco_arestas(B_bolso, pos_b, cor="#8f9bb3", largura=1.5), row=1, col=1)
fig.add_trace(go.Scatter(x=[pos_b[u][0] for u in unidades], y=[pos_b[u][1] for u in unidades], mode="markers+text",
                         text=unidades, textposition="middle left", textfont=dict(size=12, color="#0b0b0b"),
                         marker=dict(color=CINZA_NO, size=22, line=dict(color="white", width=1.5)), showlegend=False,
                         hovertext=[f"<b>{u}</b> · {TEXTO_BOLSO[k]}" for k, u in enumerate(unidades)], hoverinfo="text"), row=1, col=1)
fig.add_trace(go.Scatter(x=[pos_b[i][0] for i in itens], y=[pos_b[i][1] for i in itens], mode="markers+text",
                         text=itens, textposition="middle right", textfont=dict(size=12, color="#0b0b0b"),
                         marker=dict(color=[TIPO_COR[TIPO_BOLSO[i]] for i in itens], size=22, line=dict(color="white", width=1.5)),
                         showlegend=False, hovertext=[f"<b>{i}</b> · {TIPO_NOME[TIPO_BOLSO[i]]}" for i in itens], hoverinfo="text"), row=1, col=1)
fig.update_xaxes(range=[-0.35, 1.45], row=1, col=1)
# direita: uma trace por aresta para ter hover com o peso
for u, v, d in P_bolso.edges(data=True):
    comuns = sorted(set(B_bolso[u]) & set(B_bolso[v]))
    fig.add_trace(go.Scatter(x=[pos_p[u][0], pos_p[v][0]], y=[pos_p[u][1], pos_p[v][1]], mode="lines",
                             line=dict(color="#8f9bb3", width=2 + 3 * (d["weight"] - 1)), opacity=0.8, showlegend=False,
                             hoverinfo="text", hovertext=f"{u} · {v}<br>peso = {d['weight']} ({', '.join(comuns)})"),
                  row=1, col=2)
cores_p = [TIPO_COR[TIPO_BOLSO[n]] for n in P_bolso.nodes()]
hover_p = [f"<b>{n}</b><br>aparece em {sorted(B_bolso[n])}" for n in P_bolso.nodes()]
fig.add_trace(traco_nos(P_bolso, pos_p, cores_p, 34, hover_p, tamanho_fonte=10), row=1, col=2)
fig.update_layout(height=470, plot_bgcolor=SUPERFICIE, paper_bgcolor=SUPERFICIE, margin=dict(l=10, r=10, t=50, b=10),
                  hovermode="closest")
fig.update_xaxes(visible=False); fig.update_yaxes(visible=False)
fig.show()

print("arestas da projeção com peso:")
for u, v, d in sorted(P_bolso.edges(data=True), key=lambda e: -e[2]["weight"]):
    print(f"  {u:6s} · {v:6s}  peso {d['weight']}")

arestas da projeção com peso:
  Bruno  · UFRN    peso 2
  Carla  · Recife  peso 2
  Ana    · UFRN    peso 1
  Ana    · Recife  peso 1
  Ana    · Bruno   peso 1
  Ana    · Carla   peso 1
  Bruno  · Natal   peso 1
  Carla  · Diego   peso 1
  Carla  · Natal   peso 1
  Diego  · Recife  peso 1
  Diego  · Natal   peso 1
  Natal  · UFRN    peso 1


> **⏸️ Pare e pense.** Diego e Natal estão ligados. Volte ao texto. A frase que os liga diz que Diego *nunca esteve* em Natal. A rede está certa (os dois estão na mesma sentença) e a leitura ingênua está errada. Guarde este exemplo. Ele reaparece toda vez que alguém disser que uma aresta "significa relacionamento".

## 3.2 · O mesmo resultado por três caminhos

A projeção é uma operação tão simples que vale implementá-la de três formas e conferir que dão a mesma resposta. Cada forma ensina uma coisa diferente.

1. **Pares por unidade** (`itertools.combinations`), a forma mais direta, e a que mostra o custo $\binom{k}{2}$ por unidade.
2. **Matriz** $B^{\top}B$, a forma que liga o assunto à álgebra linear das aulas anteriores.
3. **NetworkX** (`bipartite.weighted_projected_graph`), a forma que usaremos daqui em diante.

In [9]:
# 1) pares por unidade
def coocorrencia_por_pares(itens_por_unidade):
    g = nx.Graph()
    pares_gerados = 0
    for u, lista in itens_por_unidade.items():
        presentes = sorted(set(lista))                   # presença, não número de menções
        for a, b in itertools.combinations(presentes, 2):
            pares_gerados += 1
            if g.has_edge(a, b):
                g[a][b]["weight"] += 1
            else:
                g.add_edge(a, b, weight=1)
    return g, pares_gerados

g1, pares = coocorrencia_por_pares(ITENS_BOLSO)
print(f"pares gerados: {pares} (soma de k(k-1)/2 por unidade) | arestas distintas: {g1.number_of_edges()}")

# 2) matriz de incidência e B^T B
B_mat = pd.DataFrame(0, index=unidades, columns=itens)
for u, lista in ITENS_BOLSO.items():
    B_mat.loc[f"S{u}", lista] = 1
C = B_mat.T @ B_mat
display(B_mat)
display(C)

# 3) NetworkX
g3 = bipartite.weighted_projected_graph(B_bolso, itens)

# os três caminhos coincidem?
iguais = all(C.loc[u, v] == g1[u][v]["weight"] == g3[u][v]["weight"] for u, v in g1.edges()) \
         and g1.number_of_edges() == g3.number_of_edges() == (np.triu(C.values, 1) > 0).sum()
print("✅ as três construções coincidem" if iguais else "❌ algo difere")
print("diagonal de C (unidades por item):", dict(zip(C.index, np.diag(C.values))))

pares gerados: 14 (soma de k(k-1)/2 por unidade) | arestas distintas: 12


,Ana,Bruno,Carla,Diego,Natal,Recife,UFRN
S1,1,1,0,0,0,0,1
S2,0,1,0,0,1,0,1
S3,0,0,1,0,1,0,0
S4,1,0,1,0,0,1,0
S5,0,0,1,1,0,1,0
S6,0,0,0,1,1,0,0


,Ana,Bruno,Carla,Diego,Natal,Recife,UFRN
Ana,2,1,1,0,0,1,1
Bruno,1,2,0,0,1,0,2
Carla,1,0,3,1,1,2,0
Diego,0,0,1,2,1,1,0
Natal,0,1,1,1,3,0,1
Recife,1,0,2,1,0,2,0
UFRN,1,2,0,0,1,0,2


✅ as três construções coincidem
diagonal de C (unidades por item): {'Ana': np.int64(2), 'Bruno': np.int64(2), 'Carla': np.int64(3), 'Diego': np.int64(2), 'Natal': np.int64(3), 'Recife': np.int64(2), 'UFRN': np.int64(2)}


> **⏸️ Pare e pense.** Na matriz $C$, a diagonal de `Carla` vale 3 e a de `Natal` vale 3, mas $C_{\text{Carla},\text{Natal}} = 1$. Calcule o Jaccard desse par de cabeça. Agora compare com `Diego`–`Recife`. Qual dos dois pares é "mais exclusivo"?

## 3.3 · A janela muda a rede (versão de bolso)

Fatiar de outro jeito é mudar $U$. Se a unidade passa a ser o **texto inteiro**, tudo co-ocorre com tudo e a rede vira um clique de 7 nós com 21 arestas. Se a unidade passa a ser a **palavra** (janela de tamanho 1), quase nada co-ocorre. A escolha da janela é a decisão de modelagem mais importante desta aula, e a Seção 5 mede o efeito dela em um romance.

In [10]:
def rede_de_unidades(itens_por_unidade):
    return coocorrencia_por_pares(itens_por_unidade)[0]

por_sentenca = rede_de_unidades(ITENS_BOLSO)
por_texto = rede_de_unidades({0: [i for lista in ITENS_BOLSO.values() for i in lista]})
por_par_de_sentencas = rede_de_unidades({k: ITENS_BOLSO[k] + ITENS_BOLSO[k + 1] for k in range(1, 6)})

for nome, g in [("sentença", por_sentenca), ("duas sentenças (janela deslizante)", por_par_de_sentencas), ("texto inteiro", por_texto)]:
    print(f"{nome:36s} nós={g.number_of_nodes()} arestas={g.number_of_edges():2d} densidade={nx.density(g):.2f}")

sentença                             nós=7 arestas=12 densidade=0.57
duas sentenças (janela deslizante)   nós=7 arestas=17 densidade=0.81
texto inteiro                        nós=7 arestas=21 densidade=1.00


> ✅ **O que fica.** Uma rede de co-ocorrência é a projeção de um bipartido unidades × itens, com peso igual ao número de unidades compartilhadas, ou, na forma matricial, $C = B^{\top}B$. Cada unidade com $k$ itens vira um clique de $\binom{k}{2}$ arestas. A janela define $U$, e mudar a janela muda a rede. Nada disso precisou de inteligência artificial. O que precisa é o passo que fizemos à mão, marcar os itens, e é para isso que entra a ferramenta da próxima seção.

# 4 · 🧰 A ferramenta: reconhecimento de entidades nomeadas

> 🧰 **Caixa de ferramentas.** Esta é a única seção do notebook que fala de processamento de linguagem natural, e ela existe para que a função `extrair_entidades` faça sentido. Tudo que está *dentro* do modelo (vetores de palavras, redes neurais, treinamento) fica fechado na caixa. Quem quiser abri-la tem a disciplina de PLN. Para a nossa aula basta o contrato.

**O que a ferramenta faz.** Recebe um texto e devolve os trechos que nomeiam algo do mundo (uma pessoa, um lugar, uma organização), cada um com um rótulo de tipo. O modelo em português do spaCy usa quatro rótulos, `PER`, `LOC`, `ORG` e `MISC` (miscelânea, o que não coube nos outros três).

**Como ela faz, em uma frase.** Um modelo estatístico, treinado em textos anotados por humanos (notícias e Wikipédia, sobretudo), aprende pistas como inicial maiúscula, palavras vizinhas e nomes já vistos. Disso decorrem três limitações que vamos encontrar nos dados, e que são **nossas** para resolver, não da ferramenta.

1. Ela **erra por excesso**. Palavras com inicial maiúscula no começo de frase viram "entidades" (`Vamos`, `Vi`, `Juro`).
2. Ela **confunde tipos**. Um nome que não apareceu no treinamento pode receber o rótulo errado. A versão pequena do modelo (`pt_core_news_sm`) rotula Capitu como `LOC` em 227 das 334 vezes; a versão média acerta quase sempre. Pádua, que também é uma cidade italiana, sai como `LOC` mesmo na versão média.
3. Ela **não resolve identidade**. `Bentinho`, `Bento`, `Santiago` e `Dom Casmurro` saem como quatro coisas diferentes. Juntá-las é uma decisão de modelagem.

**O contrato.** A função abaixo recebe texto e devolve uma tabela com uma linha por menção. É a única fronteira entre o NER e o resto do notebook. Se amanhã a ferramenta for outra (um modelo maior, uma lista de nomes, uma API), só esta função muda.

In [11]:
def extrair_entidades(texto, nlp=nlp):
    """Contrato da ferramenta: texto -> tabela (mencao, tipo, inicio, fim, sentenca)."""
    doc = nlp(texto)
    sentenca_de = {s.start: k for k, s in enumerate(doc.sents)}
    return pd.DataFrame([{"mencao": e.text, "tipo": e.label_, "inicio": e.start_char, "fim": e.end_char,
                          "sentenca": sentenca_de[e.sent.start]} for e in doc.ents])

texto_bolso = " ".join(TEXTO_BOLSO)
tab_bolso = extrair_entidades(texto_bolso)
tab_bolso

,mencao,tipo,inicio,fim,sentenca
0,Ana,PER,0,3,0
1,Bruno,PER,6,11,0
2,UFRN,LOC,23,27,0
3,Bruno,PER,29,34,1
4,Natal,LOC,43,48,1
5,UFRN,LOC,59,63,1
6,Carla,PER,65,70,2
7,Natal,LOC,79,84,2
8,Ana,PER,97,100,3
9,Recife,LOC,113,119,3


> 🧠 **Gestalt em ação · Similaridade.** O `displaCy` pinta cada menção com a cor do tipo. Usamos exatamente as cores da nossa paleta (`TIPO_COR`), para que "azul = pessoa, laranja = lugar" valha aqui, nas tabelas e em todas as redes que vêm depois.

In [12]:
from spacy import displacy

opcoes = {"colors": {t: c for t, c in TIPO_COR.items()}, "ents": list(TIPO_COR)}
displacy.render(nlp(texto_bolso), style="ent", jupyter=True, options=opcoes)

Compare com a marcação manual da Seção 3. A ferramenta acha **todos os trechos certos**, mas erra dois tipos (`UFRN` vira lugar e, em uma das frases, `Carla` vira lugar). Para a rede isso é irrelevante, porque a rede só usa **quais menções estão em qual unidade**. O tipo é um atributo do nó, e podemos corrigi-lo por **voto majoritário** entre as menções do mesmo nome. A célula abaixo prova o ponto, reconstruindo a rede de bolso a partir da saída do NER e comparando com a construída à mão.

In [13]:
itens_ner = tab_bolso.groupby("sentenca").mencao.apply(lambda s: sorted(set(s))).to_dict()
g_ner, _ = coocorrencia_por_pares(itens_ner)
tipo_voto = tab_bolso.groupby("mencao").tipo.agg(lambda s: s.value_counts().index[0])

mesmas_arestas = set(map(frozenset, g_ner.edges())) == set(map(frozenset, P_bolso.edges()))
mesmos_pesos = all(g_ner[u][v]["weight"] == P_bolso[u][v]["weight"] for u, v in g_ner.edges())
print("mesma rede que a marcação manual?", "✅ sim" if (mesmas_arestas and mesmos_pesos) else "❌ não")
print("tipos por voto majoritário:", tipo_voto.to_dict())
print("tipos que ainda divergem da marcação manual:", {k: v for k, v in tipo_voto.items() if TIPO_BOLSO[k] != v})

mesma rede que a marcação manual? ✅ sim
tipos por voto majoritário: {'Ana': 'PER', 'Bruno': 'PER', 'Carla': 'PER', 'Diego': 'PER', 'Natal': 'LOC', 'Recife': 'LOC', 'UFRN': 'LOC'}
tipos que ainda divergem da marcação manual: {'UFRN': 'LOC'}


> 🧭 **De volta ao grafo.** A partir daqui o NER é só um produtor de tabelas. Toda a inteligência da aula está no que fazemos com a tabela, e isso é modelagem de grafos.

> ✅ **O que fica.** NER é uma função de texto para tabela de menções. Os trechos costumam vir certos e os tipos, nem sempre. A rede depende dos trechos, não dos tipos. Identidade (quem é quem) não vem da ferramenta.

# 5 · Estudo de caso 1 · Quem aparece com quem em *Dom Casmurro*

*Dom Casmurro* (1899) tem 148 capítulos curtos, narrados em primeira pessoa por Bento Santiago, o Bentinho, já velho, relembrando a infância na Rua de Mata-cavalos, o seminário, o casamento com Capitu e a amizade com Escobar. É o romance mais discutido da literatura brasileira, e a discussão inteira gira em torno de quem esteve com quem. Ótimo material para uma rede de co-ocorrência.

O plano é o de sempre, **fatiar, extrair, ligar**, com uma etapa nova entre extrair e ligar, **resolver**, que transforma menções em nós.

## 5.1 · Fatiar

O arquivo do corpus tem os capítulos marcados por linhas `CAPÍTULO N`. O texto vem com quebras de linha "duras" no meio dos parágrafos, então normalizamos. Uma quebra simples vira espaço e uma linha em branco separa parágrafos. Isso importa porque o NER usa a vizinhança das palavras, e `José\nDias` não é `José Dias`.

In [14]:
def normalizar_paragrafos(trecho):
    trecho = re.sub(r"\n[ \t]*\n", "<<PAR>>", trecho.strip())   # linha em branco = fim de parágrafo
    trecho = re.sub(r"\s*\n\s*", " ", trecho)                    # quebra simples = espaço
    return trecho.replace("<<PAR>>", "\n\n")

def fatiar_capitulos(texto_bruto):
    partes = re.split(r"^\s*CAP[ÍI]TULO[^\n]*$", texto_bruto, flags=re.M)
    return [normalizar_paragrafos(p) for p in partes[1:]]

capitulos = fatiar_capitulos(ler_machado("romance/marm08.txt"))
titulos = [c.split("\n\n")[0][:60] for c in capitulos]        # a primeira linha de cada capítulo é o título
n_par = sum(c.count("\n\n") + 1 for c in capitulos)
print(f"{len(capitulos)} capítulos · {n_par} parágrafos · {sum(map(len, capitulos)):,} caracteres")
print("primeiros títulos:", titulos[:5])
print("\ncapítulo 1, início:\n", capitulos[0][:420], "...")

148 capítulos · 1575 parágrafos · 370,730 caracteres
primeiros títulos: ['DO TÍTULO', 'DO LIVRO', 'A DENÚNCIA', 'UM DEVER AMARÍSSIMO!', 'O AGREGADO']

capítulo 1, início:
 DO TÍTULO

Uma noite destas, vindo da cidade para o Engenho Novo, encontrei no trem da Central um rapaz aqui do bairro, que eu conheço de vista e de chapéu. Cumprimentou-me, sentou-se ao pé de mim, falou da Lua e dos ministros, e acabou recitando-me versos. A viagem era curta, e os versos pode ser que não fossem inteiramente maus. Sucedeu, porém, que, como eu estava cansado, fechei os olhos três ou quatro vezes; tant ...


## 5.2 · Extrair

`nlp.pipe` processa os 148 capítulos em lote (cerca de dez segundos aqui, talvez meio minuto no Colab). Guardamos, para cada menção, **três identificadores de unidade** ao mesmo tempo, o capítulo, o parágrafo e a sentença. Assim a Seção 5.8 poderá trocar a janela sem rodar o NER de novo.

In [15]:
def normalizar_mencao(texto):
    return re.sub(r"\s+", " ", texto).strip(" .,;:!?\"'()[]«»“”")

def tabela_mencoes(docs):
    """uma linha por menção, com o id da unidade em três níveis (capítulo, parágrafo, sentença)"""
    linhas = []
    for c, doc in enumerate(docs, start=1):
        limites = [m.start() for m in re.finditer(r"\n\s*\n", doc.text)] + [len(doc.text) + 1]
        sentenca_de = {s.start: k for k, s in enumerate(doc.sents)}
        for e in doc.ents:
            par = next(k for k, lim in enumerate(limites) if e.start_char < lim)
            k_sent = sentenca_de[e.sent.start]
            linhas.append({"capitulo": c, "paragrafo": f"{c}.{par + 1}", "sentenca": f"{c}.{k_sent}",
                           "idx_sentenca": k_sent, "mencao": normalizar_mencao(e.text), "tipo": e.label_})
    return pd.DataFrame(linhas)

t0 = time.time()
docs_dc = list(nlp.pipe(capitulos, batch_size=8))
mencoes = tabela_mencoes(docs_dc)
print(f"NER em {time.time() - t0:.1f} s · {len(mencoes)} menções · {mencoes.mencao.nunique()} formas distintas")
print(mencoes.tipo.value_counts().to_dict())
mencoes.head(8)

NER em 11.9 s · 2099 menções · 716 formas distintas
{'PER': 1361, 'LOC': 428, 'MISC': 222, 'ORG': 88}


,capitulo,paragrafo,sentenca,idx_sentenca,mencao,tipo
0,1,1.1,1.0,0,DO TÍTULO,ORG
1,1,1.2,1.0,0,Engenho Novo,LOC
2,1,1.2,1.0,0,Central,LOC
3,1,1.2,1.1,1,Lua,LOC
4,1,1.3,1.4,4,Continue,LOC
5,1,1.6,1.7,7,Dom Casmurro,MISC
6,1,1.6,1.10,10,Dom Casmurro,MISC
7,1,1.6,1.11,11,Petrópolis,LOC


## 5.3 · Olhar a lista suja antes de limpar

Uma regra de ouro em qualquer pipeline, **olhar a saída bruta**. A tabela abaixo lista as formas mais frequentes por número de capítulos em que aparecem, com o tipo majoritário. Cada patologia da ferramenta está lá, à vista.

In [16]:
def resumo_formas(df, col_unidade):
    r = (df.groupby("mencao")
           .agg(unidades=(col_unidade, "nunique"), mencoes=("mencao", "size"),
                tipo=("tipo", lambda s: s.value_counts().index[0]))
           .sort_values(["unidades", "mencoes"], ascending=False))
    return r

formas = resumo_formas(mencoes, "capitulo")
formas.head(30).T

mencao,Capitu,José Dias,Escobar,Bentinho,Deus,Justina,Cosme,Ezequiel,Céu,Europa,...,Terra,D. Fortunata,São Paulo,Manduca,Papai,Gurgel,Engenho Novo,Escritura,São José,Flamengo
unidades,91,61,36,31,28,23,22,19,19,17,...,9,8,8,7,7,7,7,7,7,6
mencoes,333,145,105,50,51,33,33,45,27,22,...,11,12,9,19,12,11,9,8,7,9
tipo,PER,PER,PER,PER,PER,PER,PER,PER,LOC,LOC,...,LOC,PER,LOC,PER,PER,PER,LOC,MISC,LOC,ORG


> 🧠 **Gestalt em ação · Similaridade + Figura e fundo.** As 30 formas mais frequentes, coloridas pelo tipo majoritário. Sem nenhum filtro, o olho já separa três grupos. Nomes de personagens (azul), lugares reais do Rio e do mundo (laranja) e um terceiro grupo misto, entidades que existem mas não são personagens (`Deus`, `Céu`, `Escritura`) e variantes ainda não unificadas do mesmo personagem (`Mamãe`, `mamãe`, `D. Glória`). Passe o mouse para ver quantas vezes cada forma recebeu cada rótulo.

In [17]:
top = formas.head(30).copy()
hover = []
for m in top.index:
    votos = mencoes[mencoes.mencao == m].tipo.value_counts().to_dict()
    hover.append(f"<b>{m}</b><br>capítulos: {top.loc[m, 'unidades']}<br>menções: {top.loc[m, 'mencoes']}<br>rótulos: {votos}")
fig = go.Figure(go.Bar(x=top.unidades, y=top.index, orientation="h", marker_color=[TIPO_COR[t] for t in top.tipo],
                       hovertext=hover, hoverinfo="text", showlegend=False))
fig.update_layout(title="As 30 formas mais frequentes (em capítulos) · cor = tipo majoritário",
                  height=720, plot_bgcolor=SUPERFICIE, paper_bgcolor=SUPERFICIE, xaxis_title="capítulos em que aparece",
                  yaxis=dict(autorange="reversed"), xaxis=dict(gridcolor=GRADE), margin=dict(l=10, r=10, t=50, b=10))
legenda_tipos(fig)
fig.show()

Três coisas para reparar na figura, uma para cada limitação listada na caixa de ferramentas.

* **Ruído de início de frase.** Um pouco mais abaixo na lista, como mostra a tabela de descartes da Seção 5.4, aparecem `Vamos`, `Vi`, `Fui` e `Juro`, verbos que viraram entidade por estarem em maiúscula no começo da frase. Todos têm uma propriedade em comum, aparecem **em minúscula** em outros lugares do texto. `Capitu` nunca aparece como `capitu`. Isso dá um filtro automático barato, o **teste da minúscula**.
* **Tipo errado.** `Pádua` (o pai de Capitu) é `LOC` na maioria das vezes, e `Mata-cavalos` (uma rua) recebe `PER` de vez em quando. Para a rede, tanto faz. Para a legenda, corrigimos com conhecimento de domínio.
* **Identidade partida.** `D. Glória`, `Mamãe`, `Dona Glória` e `Prima Glória` são a mesma pessoa. `Cosme` e `Tio Cosme` também. `Bentinho`, `Bento`, `Santiago` e `Dom Casmurro` também. Nenhum algoritmo cego resolve isso com segurança; quem leu o livro resolve em dois minutos.

## 5.4 · Resolver: de menções a nós

A resolução tem duas passagens, e a ordem importa.

1. **Dicionário de aliases**, escrito por quem conhece o texto. Ele tem prioridade absoluta, porque carrega informação que não está na estatística (que `Mamãe` é `D. Glória`, por exemplo).
2. **Peneira automática** para o que sobrou, com quatro critérios. Tipo majoritário `PER`, aparecer em pelo menos 3 capítulos, passar no teste da minúscula e ter forma de nome próprio (só palavras com inicial maiúscula, ligadas por `de`, `da`, `e`).

Não existe pipeline de texto sem uma lista como a de baixo. O que existe é pipeline que **esconde** a lista. Aqui ela fica explícita, com 16 personagens, e é o primeiro lugar a mexer se algo parecer estranho na rede.

In [18]:
ALIASES = {
    "Bentinho":      ["Bentinho", "Bento", "Bento Santiago", "Santiago", "Dom Casmurro"],
    "Capitu":        ["Capitu", "Capitolina", "Capituzinha"],
    "José Dias":     ["José Dias", "Sr. José Dias"],
    "Escobar":       ["Escobar", "Ezequiel de Sousa Escobar", "Ezequiel A. de Sousa Escobar"],
    "Ezequiel":      ["Ezequiel", "Ezequielzinho"],
    "D. Glória":     ["D. Glória", "Dona Glória", "Mamãe", "Prima Glória", "Mana Glória", "Maria da Glória"],
    "Tio Cosme":     ["Tio Cosme", "Cosme", "Mano Cosme"],
    "Prima Justina": ["Prima Justina", "Justina", "D. Justina"],
    "Padre Cabral":  ["Padre Cabral", "Cabral", "Protonotário"],
    "Sancha":        ["Sancha", "Sanchinha", "D. Sancha"],
    "Pádua":         ["Pádua", "Sr. Pádua"],
    "D. Fortunata":  ["D. Fortunata", "Fortunata"],
    "Manduca":       ["Manduca", "Coitado de Manduca"],
    "Gurgel":        ["Gurgel"],
    "Papai":         ["Papai"],
    "Imperador":     ["Imperador", "D. Pedro II"],
}
MAPA_ALIASES = {forma.lower(): canonico for canonico, formas in ALIASES.items() for forma in formas}
TITULOS_SOLTOS = {"sr", "srs", "sra", "d", "dr", "dona", "padre", "senhor", "senhora"}
RUIDO_CONHECIDO = {"medicina", "cântico", "escritura", "panegírico"}   # três rótulos PER errados que a peneira não pega
TOKEN_NOME = r"(?:[A-ZÀ-Ý][A-Za-zà-ÿ]+\.?|[A-ZÀ-Ý]\.)"          # Pedro, Glória, Sr., D.
PADRAO_NOME = re.compile(rf"^{TOKEN_NOME}(?: (?:d[aeo]s?|e|{TOKEN_NOME}|[IVX]+))*$")   # "D. Pedro II", "Rua do Ouvidor"

def palavras_minusculas(docs):
    return {tok.text for doc in docs for tok in doc if tok.is_alpha and tok.text.islower()}

def resolver(df, minusculas, col_unidade="capitulo", min_unidades=3, tipo_alvo="PER", mapa=MAPA_ALIASES):
    df = df.copy()
    df["no"] = df.mencao.str.lower().map(mapa)                       # 1) dicionário
    resto = df[df.no.isna()]
    r = resumo_formas(resto, col_unidade)                            # 2) peneira automática
    aprovadas = {m for m in r.index
                 if r.loc[m, "tipo"] == tipo_alvo and r.loc[m, "unidades"] >= min_unidades
                 and PADRAO_NOME.match(m) and m.lower() not in minusculas
                 and m.lower() not in TITULOS_SOLTOS and m.lower() not in RUIDO_CONHECIDO}
    df.loc[df.no.isna(), "no"] = resto.mencao.where(resto.mencao.isin(aprovadas))
    df["origem"] = np.where(df.mencao.str.lower().isin(mapa), "dicionário",
                            np.where(df.no.notna(), "peneira automática", "descartada"))
    return df

minusculas_dc = palavras_minusculas(docs_dc)
mencoes = resolver(mencoes, minusculas_dc)
print(mencoes.origem.value_counts().to_dict())
print("aprovadas pela peneira automática:", sorted(mencoes[mencoes.origem == "peneira automática"].no.unique()))
print("nós finais:", mencoes.no.nunique())

{'descartada': 1054, 'dicionário': 1041, 'peneira automática': 4}
aprovadas pela peneira automática: ['Desdêmona']
nós finais: 17


A peneira automática aprovou uma única forma fora do dicionário, `Desdêmona`. Não é personagem do romance, é a esposa de Otelo, na peça de Shakespeare que Bentinho vê no teatro e usa para pensar o próprio ciúme. Vamos deixá-la na rede de propósito. Ela é um lembrete de que "entidade nomeada" não é "personagem", e vai aparecer em um lugar interessante.

> **⏸️ Pare e pense.** Das 2.099 menções, metade foi descartada. Isso é ruim? Antes de responder, olhe *o que* foi descartado (a célula abaixo). Depois pense no que aconteceria se, em vez de descartar, tivéssemos mantido tudo.

In [19]:
descartadas = resumo_formas(mencoes[mencoes.origem == "descartada"], "capitulo")
print("formas descartadas mais frequentes:")
descartadas.head(24).T

formas descartadas mais frequentes:


mencao,Deus,Céu,Europa,Mata-cavalos,Glória,Terra,São Paulo,Engenho Novo,Escritura,São José,...,Vi,vim,Rua dos Inválidos,Sr,Vamos,Panegírico,Andaraí,Medicina,Senhor,Juro
unidades,28,19,17,16,10,9,8,7,7,7,...,6,6,5,5,5,4,4,4,4,4
mencoes,51,27,22,17,15,11,9,9,8,7,...,6,6,6,5,5,7,6,6,6,5
tipo,PER,LOC,LOC,LOC,LOC,LOC,LOC,LOC,MISC,LOC,...,MISC,MISC,LOC,PER,LOC,PER,LOC,PER,PER,PER


### 5.4.1 · Contração de nós, e por que ela não comuta com a projeção

Há um jeito tentador de resolver aliases **depois** de construir a rede. Construir a rede com as formas de superfície e depois **contrair** os nós de cada personagem em um só, somando os pesos das arestas. É uma operação clássica de grafos (o NetworkX tem `nx.contracted_nodes`, que junta nós mas não soma pesos). O problema é sutil e vale a pena ver com números.

Se `Mamãe` e `D. Glória` aparecem no **mesmo** capítulo que `Capitu`, a rede por formas tem duas arestas (`Mamãe`–`Capitu` e `D. Glória`–`Capitu`), cada uma contando aquele capítulo. A contração soma as duas e aquele capítulo é **contado duas vezes**. Resolver na tabela, antes de projetar, conta o capítulo uma vez, como deve. Projeção e contração **não comutam**.

In [20]:
def rede_coocorrencia(df, col_unidade, col_no="no"):
    """projeção ponderada: peso = número de unidades compartilhadas; atributo 'unidades' = frequência do nó"""
    d = df.dropna(subset=[col_no])
    g = nx.Graph()
    for no, f in d.groupby(col_no)[col_unidade].nunique().items():
        g.add_node(no, unidades=int(f))
    for _, nos in d.groupby(col_unidade)[col_no].apply(lambda s: sorted(set(s))).items():
        for a, b in itertools.combinations(nos, 2):
            if g.has_edge(a, b):
                g[a][b]["weight"] += 1
            else:
                g.add_edge(a, b, weight=1)
    return g

def contrair(g, mapa):
    """junta nós segundo o mapa forma -> canônico, somando pesos (a versão ingênua, para comparar)"""
    h = nx.Graph()
    for n, d in g.nodes(data=True):
        m = mapa.get(n, n)
        h.add_node(m, unidades=h.nodes[m]["unidades"] + d["unidades"] if m in h else d["unidades"])
    for u, v, d in g.edges(data=True):
        a, b = mapa.get(u, u), mapa.get(v, v)
        if a != b:
            h.add_edge(a, b, weight=h[a][b]["weight"] + d["weight"] if h.has_edge(a, b) else d["weight"])
    return h

# rede por formas de superfície (só as formas do dicionário), depois contraída
dic = mencoes[mencoes.origem == "dicionário"]
g_formas = rede_coocorrencia(dic.assign(no=dic.mencao), "capitulo")
g_contraida = contrair(g_formas, {m: MAPA_ALIASES[m.lower()] for m in dic.mencao.unique()})

# rede resolvida na tabela antes de projetar (o jeito certo)
g_cap = rede_coocorrencia(mencoes, "capitulo")

print("por formas:", g_formas, "| contraída:", g_contraida, "| resolvida na tabela:", g_cap)
comparacao = pd.DataFrame([{"par": f"{a} · {b}", "contração (soma)": g_contraida[a][b]["weight"], "tabela (correto)": g_cap[a][b]["weight"]}
                           for a, b in [("Capitu", "D. Glória"), ("José Dias", "Tio Cosme"), ("Capitu", "Bentinho"), ("Capitu", "José Dias")]])
comparacao

por formas: Graph with 45 nodes and 300 edges | contraída: Graph with 16 nodes and 89 edges | resolvida na tabela: Graph with 17 nodes and 93 edges


,par,contração (soma),tabela (correto)
0,Capitu · D. Glória,37,24
1,José Dias · Tio Cosme,40,22
2,Capitu · Bentinho,32,29
3,Capitu · José Dias,43,40


> 🧠 **Gestalt em ação · Similaridade + Fechamento.** À esquerda, a rede por formas de superfície, com cada forma colorida pelo personagem a que pertence. As formas do mesmo personagem têm a mesma cor e o *layout* ponderado tende a aproximá-las, porque co-ocorrem com os mesmos vizinhos. À direita, a rede resolvida. O olho "fecha" cada mancha de cor da esquerda em um único nó da direita.

In [21]:
CAT16 = CAT + ["#7f4f24", "#00a2c7", "#c04ab8", "#8c8c00", "#ff8fab", "#4d6b9a", "#a0522d", "#5fb0ff"]   # 16 matizes distintas
cor_canonico = {c: CAT16[k] for k, c in enumerate(ALIASES)}
pos_f = nx.spring_layout(g_formas, seed=11, weight="weight", k=1.1)
pos_c = nx.spring_layout(g_cap, seed=11, weight="weight", k=1.2)

fig = make_subplots(rows=1, cols=2, subplot_titles=(f"por formas de superfície · {g_formas.number_of_nodes()} nós (rótulo se ≥ 2 capítulos)",
                                                    f"resolvida · {g_cap.number_of_nodes()} nós"), horizontal_spacing=0.03)
for t in tracos_arestas_ponderadas(g_formas, pos_f, opacidade=0.3):
    fig.add_trace(t, row=1, col=1)
nos_f = list(g_formas.nodes())
fig.add_trace(go.Scatter(x=[pos_f[n][0] for n in nos_f], y=[pos_f[n][1] for n in nos_f], mode="markers+text",
                         text=[n if g_formas.nodes[n]["unidades"] >= 2 else "" for n in nos_f], textposition="top center",
                         textfont=dict(size=9, color="#0b0b0b"), showlegend=False,
                         hovertext=[f"<b>{n}</b> → {MAPA_ALIASES[n.lower()]}<br>{g_formas.nodes[n]['unidades']} capítulos" for n in nos_f], hoverinfo="text",
                         marker=dict(color=[cor_canonico[MAPA_ALIASES[n.lower()]] for n in nos_f],
                                     size=tamanho_por([g_formas.nodes[n]["unidades"] for n in nos_f], 8, 30), line=dict(color="white", width=1.5))),
              row=1, col=1)
for t in tracos_arestas_ponderadas(g_cap, pos_c, opacidade=0.3):
    fig.add_trace(t, row=1, col=2)
fig.add_trace(traco_nos(g_cap, pos_c, [cor_canonico.get(n, CINZA_NO) for n in g_cap.nodes()],
                        tamanho_por(nx.get_node_attributes(g_cap, "unidades").values(), 10, 40),
                        [f"<b>{n}</b><br>{g_cap.nodes[n]['unidades']} capítulos" for n in g_cap.nodes()],
                        rotulos=True, posicao_texto="top center", tamanho_fonte=10), row=1, col=2)
fig.update_layout(height=620, plot_bgcolor=SUPERFICIE, paper_bgcolor=SUPERFICIE, margin=dict(l=10, r=10, t=50, b=10), hovermode="closest")
fig.update_xaxes(visible=False); fig.update_yaxes(visible=False)
fig.show()

## 5.5 · Ligar: a rede de personagens por capítulo

Agora a rede prometida na abertura, a versão brasileira da rede de Knuth, com os pesos à vista. Tamanho do nó é o número de capítulos em que o personagem aparece; largura da aresta é o número de capítulos compartilhados. O *layout* de molas usa o peso, então pares que se encontram muito ficam perto.

> 🧠 **Gestalt em ação · Proximidade.** Repare no triângulo Capitu · José Dias · Bentinho no centro, e nas figuras que se afastam para a borda. A distância no desenho não é decoração, é a força da co-ocorrência.

In [22]:
def hover_no(g, n, extras=None):
    txt = f"<b>{n}</b><br>capítulos: {g.nodes[n]['unidades']}<br>grau: {g.degree(n)}<br>força: {g.degree(n, weight='weight')}"
    if extras:
        for k, v in extras.items():
            txt += f"<br>{k}: {v[n]}"
    return txt

def desenhar_rede(g, pos, titulo, cores=None, altura=600, rotulos=True, tam=(10, 42), extras=None, cor_aresta="#8f9bb3"):
    nos = list(g.nodes())
    cores = cores if cores is not None else [AZUIS[5]] * len(nos)
    tracos = tracos_arestas_ponderadas(g, pos, cor=cor_aresta)
    tracos.append(traco_nos(g, pos, cores, tamanho_por([g.nodes[n]["unidades"] for n in nos], *tam),
                            [hover_no(g, n, extras) for n in nos], rotulos=rotulos, posicao_texto="top center", tamanho_fonte=11))
    return figura_rede(tracos, titulo, altura=altura)

pos_cap = nx.spring_layout(g_cap, seed=2026, weight="weight", k=1.4)
fig = desenhar_rede(g_cap, pos_cap, "Dom Casmurro · quem aparece com quem, por capítulo · largura = capítulos compartilhados")
fig.show()
print(g_cap, "| densidade", round(nx.density(g_cap), 3))

Graph with 17 nodes and 93 edges | densidade 0.684


## 5.6 · Ler a rede com o que já sabemos

Quatro medidas das aulas anteriores, e uma pergunta antes de olhar a tabela.

> **⏸️ Pare e pense.** A densidade é 0,68. Em uma rede tão densa, o **grau** ainda diz alguma coisa? Qual medida você usaria no lugar dele?

Para a intermediação em rede ponderada, lembre que o NetworkX interpreta o atributo de peso como **distância**. Peso alto aqui significa "perto", então passamos `1/peso`.

In [24]:
for u, v, d in g_cap.edges(data=True):
    d["dist"] = 1.0 / d["weight"]

def tabela_medidas(g):
    return (pd.DataFrame({
        "capítulos": pd.Series(nx.get_node_attributes(g, "unidades")),
        "grau": pd.Series(dict(g.degree())),
        "força": pd.Series(dict(g.degree(weight="weight"))),
        "intermediação (1/peso)": pd.Series(nx.betweenness_centrality(g, weight="dist")).round(3),
        "core": pd.Series(nx.core_number(g)),
        "clustering": pd.Series(nx.clustering(g)).round(2),
    }).sort_values("força", ascending=False))

medidas_cap = tabela_medidas(g_cap)
medidas_cap

,capítulos,grau,força,intermediação (1/peso),core,clustering
Capitu,91,16,241,0.771,9,0.64
José Dias,61,14,196,0.167,9,0.78
Bentinho,35,15,133,0.000,9,0.70
D. Glória,30,13,122,0.000,9,0.82
Prima Justina,29,15,121,0.000,9,0.70
Escobar,36,12,117,0.000,8,0.79
Tio Cosme,26,14,106,0.000,9,0.78
Ezequiel,19,11,55,0.000,8,0.82
Pádua,13,11,54,0.000,9,0.91
Sancha,19,9,53,0.000,8,0.97


Três leituras, da mais óbvia para a menos.

* **Grau saturou, força não.** Oito dos 17 personagens têm grau 12 ou mais (a rede é quase completa), mas a força vai de 241 a 5. Em redes de co-ocorrência, **força é a medida de primeira leitura**. É a mesma lição da aula de Hubs, olhando o dado certo.
* **Capitu concentra a intermediação ponderada.** Quase todos os caminhos "mais leves" passam por ela, porque as arestas mais pesadas da rede são as dela. É o centro de gravidade do romance, o que ninguém que leu o livro vai contestar.
* **Bentinho aparece menos do que deveria.** Ele é o narrador e está em todos os 148 capítulos, mas só é *nomeado* em 35. A ferramenta conta nomes, não presenças. Um narrador em primeira pessoa é um **nó fantasma**. Quando a rede é a do narrador, a rede que vemos é a das pessoas que ele decide nomear. (O Desafio 5 compara com um romance narrado em terceira pessoa.)

> 🧠 **Gestalt em ação · Figura e fundo.** Abaixo, grau no eixo x e força no eixo y. Se as duas medidas dissessem a mesma coisa, os pontos formariam uma curva crescente. Em vermelho, quem tem grau alto e força baixa. É o caso de Papai, o pai de Bentinho, morto antes de a história começar. Ele é lembrado ao lado de quase todo mundo (grau 12), mas em poucos capítulos (força 30). Está "em todos os círculos" sem estar muito com ninguém.

In [25]:
m = medidas_cap.copy()
lim_grau = m.grau.quantile(0.6)
m["destaque"] = (m.grau >= lim_grau) & (m.força <= m.força.median())
# rótulos alternados (em cima / embaixo) quando dois personagens têm o mesmo grau, para não sobrepor texto
posicao = {}
for grau, grupo in m.groupby("grau"):
    for k, nome in enumerate(grupo.sort_values("força").index):
        posicao[nome] = ["top center", "bottom center", "middle right"][k % 3]
fig = go.Figure()
for flag, cor, nome in [(False, AZUIS[5], "demais"), (True, DESTAQUE, "grau alto, força baixa")]:
    sub = m[m.destaque == flag]
    fig.add_scatter(x=sub.grau, y=sub.força, mode="markers+text", text=sub.index, textposition=[posicao[n] for n in sub.index], name=nome,
                    marker=dict(color=cor, size=12, line=dict(color="white", width=1)),
                    hovertemplate="<b>%{text}</b><br>grau %{x}<br>força %{y}<extra></extra>")
fig.update_layout(title="Grau × força · cada ponto é um personagem", xaxis_title="grau (com quantos aparece)",
                  yaxis_title="força (soma dos capítulos compartilhados)", height=460, plot_bgcolor=SUPERFICIE,
                  paper_bgcolor=SUPERFICIE, legend=dict(orientation="h", y=1.08, x=0), xaxis=dict(gridcolor=GRADE), yaxis=dict(gridcolor=GRADE))
fig.show()

## 5.7 · O esqueleto: subindo o limiar de peso

Uma rede densa esconde a estrutura na distribuição dos pesos. O jeito mais simples de revelá-la é **subir um limiar** e manter só as arestas com peso maior ou igual a ele. As arestas fracas somem primeiro, os nós que ficam isolados apagam, e o que resta no fim é o **esqueleto** do romance.

> 🧠 **Gestalt em ação · Destino comum + Fechamento.** Arraste o controle devagar. As arestas somem em grupos (mesmo peso, mesmo destino) e os nós que perdem todas as arestas ficam cinza. Note em que limiar a rede se parte pela primeira vez e quais pares sobrevivem por último.

In [26]:
def animacao_limiar(g, pos, limiares, titulo="Esqueleto por limiar", altura=600):
    nos = list(g.nodes())
    tam = tamanho_por([g.nodes[n]["unidades"] for n in nos], 10, 42)
    cortes = cortes_de_peso(g)                      # classes de largura fixas para todos os quadros
    frames, passos = [], []
    for w in limiares:
        h = nx.Graph()
        h.add_nodes_from(g.nodes(data=True))
        h.add_edges_from((u, v, d) for u, v, d in g.edges(data=True) if d["weight"] >= w)
        ativos = {n for n in nos if h.degree(n) > 0}
        comps = nx.number_connected_components(h.subgraph(ativos)) if ativos else 0
        cores = [AZUIS[5] if n in ativos else CINZA_NO for n in nos]
        opac = [1.0 if n in ativos else 0.35 for n in nos]
        dados = tracos_arestas_ponderadas(h, pos, cor="#8f9bb3", cortes=cortes)      # sempre 4 traces
        dados.append(traco_nos(g, pos, cores, tam, [hover_no(h, n) for n in nos], rotulos=True, opacidade=opac,
                               posicao_texto="top center"))
        legenda = f"peso ≥ {w} · {h.number_of_edges()} arestas · {len(ativos)} nós ativos · {comps} componente(s)"
        frames.append(go.Frame(name=str(w), data=dados, layout=go.Layout(title=dict(text=f"{titulo} · {legenda}"))))
        passos.append(dict(method="animate", label=str(w),
                           args=[[str(w)], dict(mode="immediate", frame=dict(duration=0, redraw=True), transition=dict(duration=150))]))
    fig = figura_rede(frames[0].data, frames[0].layout.title.text, altura=altura)
    fig.frames = frames
    fig.update_layout(sliders=[dict(active=0, steps=passos, x=0.02, len=0.96, y=-0.02,
                                    currentvalue=dict(prefix="peso mínimo: ", font=dict(size=13)))])
    return fig

limiares = [1, 2, 3, 5, 8, 10, 12, 15, 18, 21, 24, 28, 32, 36, 40]
animacao_limiar(g_cap, pos_cap, limiares, "Dom Casmurro · esqueleto por limiar").show()

In [27]:
pesos = sorted(((d["weight"], u, v) for u, v, d in g_cap.edges(data=True)), reverse=True)
print("as dez arestas mais pesadas (capítulos compartilhados):")
for w, u, v in pesos[:10]:
    print(f"  {w:3d}  {u} · {v}")

as dez arestas mais pesadas (capítulos compartilhados):
   40  Capitu · José Dias
   32  Capitu · Escobar
   29  Bentinho · Capitu
   26  Bentinho · José Dias
   24  Capitu · D. Glória
   23  José Dias · Prima Justina
   23  D. Glória · José Dias
   22  José Dias · Tio Cosme
   21  Capitu · Prima Justina
   17  Capitu · Tio Cosme


> **⏸️ Pare e pense.** As três arestas mais pesadas são Capitu · José Dias, Capitu · Escobar e Bentinho · Capitu. Duas observações, uma técnica e uma literária. A técnica, José Dias (o "agregado" da casa) está em tudo porque tem função de **conector** na narrativa, é ele quem leva e traz notícias. A literária, o par Capitu · Escobar compartilha 32 capítulos. A rede não julga ninguém, ela conta. Mas conta exatamente o que o romance passa 148 capítulos insinuando.

## 5.8 · A janela muda a rede

Mesmas menções, mesmos nós, três janelas. Como guardamos os três identificadores de unidade na tabela, trocar a janela custa uma linha.

> 🧠 **Gestalt em ação · Similaridade (posições fixas).** Os três painéis usam **as mesmas posições** de nó, calculadas na rede por capítulo. Assim o olho compara estrutura, não *layout*. O que muda de painel para painel é só o que a janela deixou passar.

In [28]:
redes = {"capítulo": g_cap,
         "parágrafo": rede_coocorrencia(mencoes, "paragrafo"),
         "sentença": rede_coocorrencia(mencoes, "sentenca")}

linhas = []
for nome, g in redes.items():
    top_w, top_u, top_v = max((d["weight"], u, v) for u, v, d in g.edges(data=True))
    linhas.append({"janela": nome, "unidades": mencoes[{"capítulo": "capitulo", "parágrafo": "paragrafo", "sentença": "sentenca"}[nome]].nunique(),
                   "nós": g.number_of_nodes(), "arestas": g.number_of_edges(), "densidade": round(nx.density(g), 2),
                   "grau médio": round(2 * g.number_of_edges() / g.number_of_nodes(), 1),
                   "clustering médio": round(nx.average_clustering(g), 2), "componentes": nx.number_connected_components(g),
                   "aresta mais pesada": f"{top_u} · {top_v} ({top_w})"})
pd.DataFrame(linhas).set_index("janela")

,unidades,nós,arestas,densidade,grau médio,clustering médio,componentes,aresta mais pesada
janela,,,,,,,,
capítulo,147,17,93,0.68,10.9,0.86,1,Capitu · José Dias (40)
parágrafo,978,17,63,0.46,7.4,0.80,1,Capitu · José Dias (33)
sentença,1595,17,45,0.33,5.3,0.58,2,José Dias · Tio Cosme (13)


In [29]:
fig = make_subplots(rows=1, cols=3, subplot_titles=[f"{n} · {g.number_of_edges()} arestas" for n, g in redes.items()], horizontal_spacing=0.02)
for k, (nome, g) in enumerate(redes.items(), start=1):
    for t in tracos_arestas_ponderadas(g, pos_cap, cor="#8f9bb3", opacidade=0.5):
        fig.add_trace(t, row=1, col=k)
    nos = list(g.nodes())
    cores = [AZUIS[5] if g.degree(n) > 0 else CINZA_NO for n in nos]
    fig.add_trace(traco_nos(g, pos_cap, cores, tamanho_por([g.nodes[n]["unidades"] for n in nos], 8, 30),
                            [hover_no(g, n) for n in nos], rotulos=True, posicao_texto="top center", tamanho_fonte=9), row=1, col=k)
fig.update_layout(height=490, plot_bgcolor=SUPERFICIE, paper_bgcolor=SUPERFICIE, margin=dict(l=10, r=10, t=80, b=10), hovermode="closest",
                  title=dict(text="A mesma história, três janelas · posições fixas", x=0.02, y=0.97))
fig.update_xaxes(visible=False); fig.update_yaxes(visible=False)
fig.show()

Do capítulo para a sentença a densidade cai de 0,68 para 0,33, o clustering cai e a rede se parte em dois componentes (Manduca fica sozinho, porque nunca divide uma frase com ninguém do elenco). Isso é geral. **Janela grande** produz redes densas, com cliques grandes e pouca informação estrutural, e a informação migra para os pesos. **Janela pequena** produz redes esparsas, com estrutura visível, ao custo de perder relações que o texto expressa em frases separadas.

Repare também na aresta mais pesada. Por capítulo, Capitu · José Dias, com folga. Por sentença, um empate triplo, José Dias · Tio Cosme, Capitu · Sancha e Capitu · Escobar. A janela não muda só a topologia, muda **quem parece mais próximo de quem**. Nenhuma das duas está errada. Elas medem coisas diferentes, "aparecer no mesmo episódio" e "aparecer na mesma frase".

> **⏸️ Pare e pense.** Se você quisesse uma rede de **interações** (quem fala com quem) em vez de co-presença, qual janela escolheria? E o que ainda faltaria, mesmo com a janela ideal? (Pista, releia a Seção 2.4.)

## 5.9 · Quem nunca se encontra

Uma rede densa também se lê pelo que **falta**. Entre os personagens presentes em pelo menos 7 capítulos, quais pares nunca dividem um capítulo?

In [30]:
frequentes = [n for n, f in sorted(nx.get_node_attributes(g_cap, "unidades").items(), key=lambda kv: -kv[1]) if f >= 7]
nunca = [(a, b) for a, b in itertools.combinations(frequentes, 2) if not g_cap.has_edge(a, b)]
primeiro_cap = mencoes.dropna(subset=["no"]).groupby("no").capitulo.min()
print(f"{len(frequentes)} personagens frequentes · {len(nunca)} pares que nunca dividem um capítulo\n")
for a, b in nunca:
    print(f"  {a:14s} (entra no cap. {primeiro_cap[a]:3d}) · {b:14s} (entra no cap. {primeiro_cap[b]:3d})")

15 personagens frequentes · 24 pares que nunca dividem um capítulo

  José Dias      (entra no cap.   3) · Manduca        (entra no cap.  84)
  Escobar        (entra no cap.  56) · Padre Cabral   (entra no cap.  11)
  Escobar        (entra no cap.  56) · D. Fortunata   (entra no cap.  16)
  Escobar        (entra no cap.  56) · Manduca        (entra no cap.  84)
  D. Glória      (entra no cap.   3) · Gurgel         (entra no cap.  70)
  D. Glória      (entra no cap.   3) · Manduca        (entra no cap.  84)
  Tio Cosme      (entra no cap.   3) · Manduca        (entra no cap.  84)
  Ezequiel       (entra no cap. 108) · Padre Cabral   (entra no cap.  11)
  Ezequiel       (entra no cap. 108) · Pádua          (entra no cap.   3)
  Ezequiel       (entra no cap. 108) · D. Fortunata   (entra no cap.  16)
  Ezequiel       (entra no cap. 108) · Manduca        (entra no cap.  84)
  Sancha         (entra no cap.  42) · Padre Cabral   (entra no cap.  11)
  Sancha         (entra no cap.  42) · Pádua

As ausências contam a **cronologia** do romance. Ezequiel, o filho, nasce quando Pádua, Padre Cabral e D. Fortunata já saíram de cena; Sancha, a amiga de Capitu, nunca cruza com o círculo da infância. Manduca, o vizinho doente, aparece em sete capítulos e não divide nenhum deles com dez dos quinze personagens frequentes. A rede de co-ocorrência é estática, mas o capítulo de entrada de cada personagem já sugere o filme por trás da foto. O Desafio 4 transforma a foto em filme.

> ✅ **O que fica.** Resolver identidade é modelagem, não pré-processamento, e deve ser feito na tabela, antes de projetar (projeção e contração não comutam). Em redes de co-ocorrência com elenco pequeno, o grau satura e a leitura passa pela **força** e pelo **esqueleto** por limiar. A janela decide densidade, clustering, componentes e até qual par parece mais próximo. E um narrador em primeira pessoa é um nó fantasma, a rede é a das pessoas que ele nomeia.

# 6 · Estudo de caso 2 · *A Semana*, o mundo real em 244 crônicas

O romance tinha um elenco fechado, um narrador e capítulos. Agora mudamos os três parâmetros de uma vez. *A Semana* é a coluna que Machado assinou na *Gazeta de Notícias* entre 1892 e 1897 (com dois textos soltos de 1900), comentando o noticiário da semana, a política da República recém-proclamada, a Rua do Ouvidor, a guerra entre China e Japão, Canudos, e misturando tudo com Homero, Shakespeare e a Bíblia. Isso muda a rede em três aspectos que a Seção 2 anunciou.

* A **unidade** passa a ser o **documento** (uma crônica), e documentos têm tamanhos muito diferentes.
* Os **itens** têm **tipos** misturados, pessoas, lugares, organizações e obras.
* Há uma **data** em cada unidade, o que permite ver a rede mudar no tempo.

## 6.1 · Fatiar

No arquivo, cada crônica começa com a data em um parágrafo isolado (`24 de abril`) e o ano aparece sozinho antes do primeiro texto de cada ano. Como algumas datas vêm quebradas em duas linhas (`5\nde fevereiro`), normalizamos os parágrafos **antes** de procurar as datas.

In [31]:
MESES = r"(?:janeiro|fevereiro|março|abril|maio|junho|julho|agosto|setembro|outubro|novembro|dezembro)"

def fatiar_cronicas(texto_bruto):
    paragrafos = [re.sub(r"\s*\n\s*", " ", p).strip() for p in re.split(r"\n[ \t]*\n", texto_bruto)]
    ano, cronicas = None, []
    for p in paragrafos:
        if not p:
            continue
        if re.fullmatch(r"(18\d\d|1900)", p):                       # linha só com o ano
            ano = int(p)
        elif re.fullmatch(rf"\d{{1,2}} de {MESES}", p, flags=re.I): # linha só com a data: começa uma crônica
            cronicas.append({"id": len(cronicas) + 1, "ano": ano, "data": f"{p} de {ano}", "paragrafos": []})
        elif cronicas:
            cronicas[-1]["paragrafos"].append(p)
    df = pd.DataFrame(cronicas)
    df["texto"] = df.paragrafos.map("\n\n".join)
    df["caracteres"] = df.texto.str.len()
    return df.drop(columns="paragrafos")

cronicas = fatiar_cronicas(ler_machado("cronica/macr12.txt"))
print(f"{len(cronicas)} crônicas · {cronicas.caracteres.sum():,} caracteres · de {cronicas.data.iloc[0]} a {cronicas.data.iloc[-1]}")
print("por ano:", cronicas.ano.value_counts().sort_index().to_dict())
print("\n", cronicas.data.iloc[0], "·", cronicas.texto.iloc[0][:300], "...")

244 crônicas · 1,535,878 caracteres · de 24 de abril de 1892 a 11 de novembro de 1900
por ano: {1892: 37, 1893: 44, 1894: 52, 1895: 50, 1896: 51, 1897: 8, 1900: 2}

 24 de abril de 1892 · Na segunda feira da semana que findou, acordei cedo, pouco depois das galinhas, e dei-me ao gosto de propor a mim mesmo um problema. Verdadeiramente era uma charada; mas o nome de problema dá dignidade, e excita para logo a atenção dos leitores austeros. Sou como as atrizes, que já não fazem benefíc ...


## 6.2 · Extrair e resolver, agora sem dicionário

Com 244 documentos e milhares de formas distintas, não dá para escrever um dicionário de aliases (e não temos "quem leu o livro", porque o assunto é o noticiário de 1890). Vamos usar **só a peneira automática**, agora para todos os tipos, com um cuidado a mais. Formas que diferem apenas por maiúsculas (`Rua do Ouvidor` e `rua do Ouvidor`) são fundidas, ficando com a grafia mais frequente e o tipo majoritário.

O NER leva em torno de um minuto (o corpus tem 1,5 milhão de caracteres).

In [32]:
t0 = time.time()
docs_cr = list(nlp.pipe(cronicas.texto.tolist(), batch_size=8))
mencoes_cr = pd.DataFrame([{"cronica": cid, "mencao": normalizar_mencao(e.text), "tipo": e.label_}
                           for cid, doc in zip(cronicas.id, docs_cr) for e in doc.ents])
minusculas_cr = palavras_minusculas(docs_cr)
print(f"NER em {time.time() - t0:.0f} s · {len(mencoes_cr)} menções · {mencoes_cr.mencao.nunique()} formas distintas")

def resolver_sem_dicionario(df, minusculas, col_unidade, min_unidades=3):
    df = df.copy()
    df["chave"] = df.mencao.str.lower()
    r = df.groupby("chave").agg(forma=("mencao", lambda s: s.value_counts().index[0]),
                                tipo=("tipo", lambda s: s.value_counts().index[0]),
                                unidades=(col_unidade, "nunique"), mencoes=("mencao", "size"))
    aprovadas = r[(r.unidades >= min_unidades) & r.forma.map(lambda m: bool(PADRAO_NOME.match(m)))
                  & ~r.index.isin(minusculas) & ~r.index.isin(TITULOS_SOLTOS) & (r.forma.str.len() > 2)]
    df["no"] = df.chave.map(aprovadas.forma)
    return df.drop(columns="chave"), aprovadas.set_index("forma").sort_values("unidades", ascending=False)

mencoes_cr, nos_cr = resolver_sem_dicionario(mencoes_cr, minusculas_cr, "cronica", min_unidades=3)
tipo_cr = nos_cr.tipo.to_dict()
print(f"{len(nos_cr)} nós aprovados:", nos_cr.tipo.value_counts().to_dict())

NER em 42 s · 7837 menções · 3564 formas distintas
304 nós aprovados: {'PER': 145, 'LOC': 131, 'MISC': 17, 'ORG': 11}


> 🧠 **Gestalt em ação · Similaridade.** Quatro painéis, um por tipo, sempre com a mesma cor do tipo. Já dá para ver os dois mundos que convivem na coluna, o **noticiário** (Rio de Janeiro, Inglaterra, Câmara dos Deputados, Japão) e a **biblioteca** (Homero, Shakespeare, Camões, Hamlet). Passe o mouse para ver em quantas crônicas cada um aparece.

In [33]:
fig = make_subplots(rows=1, cols=4, subplot_titles=[f"{TIPO_NOME[t]} ({(nos_cr.tipo == t).sum()})" for t in TIPO_COR], horizontal_spacing=0.09)
for k, t in enumerate(TIPO_COR, start=1):
    sub = nos_cr[nos_cr.tipo == t].head(14)
    fig.add_trace(go.Bar(x=sub.unidades, y=sub.index, orientation="h", marker_color=TIPO_COR[t], showlegend=False,
                         hovertemplate="<b>%{y}</b><br>%{x} crônicas<extra></extra>"), row=1, col=k)
    fig.update_yaxes(autorange="reversed", row=1, col=k)
fig.update_layout(height=490, plot_bgcolor=SUPERFICIE, paper_bgcolor=SUPERFICIE, margin=dict(l=10, r=10, t=80, b=10),
                  title=dict(text="As entidades mais frequentes de A Semana, por tipo (em crônicas)", x=0.02, y=0.97))
fig.update_xaxes(gridcolor=GRADE)
fig.show()

Sem dicionário, sobra ruído. `José Rodrigues` é um correspondente fictício que assina cartas ao cronista; `Câmara dos Deputados` e `Conselho Municipal` saíram como lugar e `Jornal do Comércio` como "outro", quando os três são organizações; `Sarah` e `Sarah Bernhardt` são a mesma atriz. Vamos conviver com esse ruído e, na Seção 6.4, ver que a própria rede denuncia parte dos erros.

## 6.3 · Ligar pelo bipartido, e o artefato do clique

Desta vez montamos o bipartido explicitamente e projetamos com o NetworkX. O bipartido tem 244 nós de um lado e os nós aprovados do outro.

In [34]:
presencas = mencoes_cr.dropna(subset=["no"])[["cronica", "no"]].drop_duplicates()
docs_ids = [f"c{i}" for i in cronicas.id]
entidades = sorted(nos_cr.index)

B_cr = nx.Graph()
B_cr.add_nodes_from(docs_ids, bipartite=0)
B_cr.add_nodes_from(entidades, bipartite=1)
B_cr.add_edges_from((f"c{c}", n) for c, n in zip(presencas.cronica, presencas.no))

G_cr = bipartite.weighted_projected_graph(B_cr, entidades)
nx.set_node_attributes(G_cr, {n: B_cr.degree(n) for n in entidades}, "unidades")
nx.set_node_attributes(G_cr, tipo_cr, "tipo")

k_doc = pd.Series({d: B_cr.degree(d) for d in docs_ids})            # entidades por crônica
pares_doc = k_doc * (k_doc - 1) // 2                                 # pares que cada crônica gera
print("bipartido:", B_cr)
print("projeção :", G_cr, "· densidade", round(nx.density(G_cr), 3), "· componentes", nx.number_connected_components(G_cr))
print(f"entidades por crônica: mediana {k_doc.median():.0f}, máximo {k_doc.max()} · pares gerados no total: {pares_doc.sum()} · arestas distintas: {G_cr.number_of_edges()}")
print(f"as 10% de crônicas mais 'cheias' geram {pares_doc.sort_values(ascending=False).head(len(k_doc) // 10).sum() / pares_doc.sum():.0%} dos pares")
print("degenerescência da projeção:", max(nx.core_number(G_cr).values()))

bipartido: Graph with 548 nodes and 1763 edges
projeção : Graph with 304 nodes and 6072 edges · densidade 0.132 · componentes 1
entidades por crônica: mediana 7, máximo 20 · pares gerados no total: 7214 · arestas distintas: 6072
as 10% de crônicas mais 'cheias' geram 35% dos pares
degenerescência da projeção: 25


> 🧠 **Gestalt em ação · Continuidade.** À esquerda, quantas entidades cada crônica tem. À direita, quantos pares cada uma gera, $k(k-1)/2$. A forma da curva da direita é a da esquerda **ao quadrado**. Poucas crônicas cheias respondem por uma fatia desproporcional das arestas.

In [35]:
ordem = k_doc.sort_values(ascending=False)
datas = cronicas.set_index(cronicas.id.map(lambda i: f"c{i}")).data
fig = make_subplots(rows=1, cols=2, subplot_titles=("entidades por crônica (k)", "pares gerados por crônica, k(k−1)/2"))
fig.add_trace(go.Bar(x=list(range(len(ordem))), y=ordem.values, marker_color=AZUIS[4], hovertext=[f"{datas[d]}<br>k = {ordem[d]}" for d in ordem.index],
                     hoverinfo="text", showlegend=False), row=1, col=1)
fig.add_trace(go.Bar(x=list(range(len(ordem))), y=pares_doc[ordem.index].values, marker_color=AZUIS[6],
                     hovertext=[f"{datas[d]}<br>{pares_doc[d]} pares" for d in ordem.index], hoverinfo="text", showlegend=False), row=1, col=2)
fig.update_layout(height=380, plot_bgcolor=SUPERFICIE, paper_bgcolor=SUPERFICIE, margin=dict(l=10, r=10, t=50, b=10), bargap=0.05)
fig.update_xaxes(title_text="crônicas, da mais cheia para a mais vazia", gridcolor=GRADE)
fig.update_yaxes(gridcolor=GRADE)
fig.show()

A projeção tem mais de seis mil arestas para 305 nós e uma degenerescência alta. Nada disso é "estrutura do mundo de Machado". É o **artefato do clique**. Cada crônica com 20 entidades vira um $K_{20}$ na projeção, e cliques de crônicas diferentes, sobrepostos em alguns nós (Rio de Janeiro, Europa, Inglaterra), formam um núcleo denso onde o k-core não mede coesão, mede **tamanho de documento**. É a mesma armadilha dos artigos com centenas de autores na aula de k-core, e a mesma resposta serve, olhar os pesos e normalizar.

## 6.4 · Contagem ou Jaccard?

Para cada aresta temos duas medidas. A **contagem** (crônicas compartilhadas) e o **Jaccard** (compartilhadas divididas pela união). O gráfico abaixo coloca uma contra a outra.

> 🧠 **Gestalt em ação · Figura e fundo.** A nuvem cinza é a maioria das arestas, contagem baixa e Jaccard baixo. Em vermelho, as arestas com Jaccard ≥ 0,4 e contagem ≥ 3, pares que aparecem **quase só juntos**. Passe o mouse. Alguns são pares reais do noticiário (China · Japão), alguns são pares da biblioteca (Adão · Eva, Aquiles · Heitor) e ao menos um é um **alias que a peneira não pegou** (Sarah · Sarah Bernhardt). Jaccard alto entre duas formas parecidas é um detector de aliases gratuito.

In [36]:
J_cr = bipartite.overlap_weighted_projected_graph(B_cr, entidades, jaccard=True)
arestas = pd.DataFrame([{"u": u, "v": v, "contagem": G_cr[u][v]["weight"], "jaccard": round(J_cr[u][v]["weight"], 3)}
                        for u, v in G_cr.edges()])
arestas["exclusivo"] = (arestas.jaccard >= 0.4) & (arestas.contagem >= 3)

rng = np.random.default_rng(1)
fig = go.Figure()
for flag, cor, nome, tam in [(False, CINZA_NO, "demais arestas", 5), (True, DESTAQUE, "Jaccard ≥ 0,4 e contagem ≥ 3", 10)]:
    sub = arestas[arestas.exclusivo == flag].sort_values("jaccard")
    fig.add_scatter(x=sub.contagem + rng.uniform(-0.25, 0.25, len(sub)), y=sub.jaccard, name=nome,
                    mode="markers+text" if flag else "markers",
                    text=[f"{u} · {v}" if j >= 0.5 else "" for u, v, j in zip(sub.u, sub.v, sub.jaccard)] if flag else None,
                    textposition=[["middle right", "middle left", "top center"][k % 3] for k in range(len(sub))], textfont=dict(size=9),
                    marker=dict(color=cor, size=tam, opacity=0.75 if flag else 0.35, line=dict(color="white", width=0.5)),
                    customdata=sub[["u", "v", "contagem"]].values,
                    hovertemplate="%{customdata[0]} · %{customdata[1]}<br>contagem %{customdata[2]} · Jaccard %{y}<extra></extra>")
fig.update_layout(title="Cada ponto é uma aresta · contagem × Jaccard", xaxis_title="contagem (crônicas compartilhadas)", yaxis_title="Jaccard",
                  height=460, plot_bgcolor=SUPERFICIE, paper_bgcolor=SUPERFICIE, legend=dict(orientation="h", y=1.08, x=0),
                  xaxis=dict(gridcolor=GRADE, dtick=1), yaxis=dict(gridcolor=GRADE))
fig.show()

lado_a_lado = pd.concat([arestas.sort_values("contagem", ascending=False).head(10)[["u", "v", "contagem", "jaccard"]].reset_index(drop=True),
                         arestas[arestas.contagem >= 3].sort_values("jaccard", ascending=False).head(10)[["u", "v", "contagem", "jaccard"]].reset_index(drop=True)],
                        axis=1, keys=["top 10 por contagem", "top 10 por Jaccard (contagem ≥ 3)"])
lado_a_lado

top 10 por contagem                                       \
                    u                   v contagem jaccard   
0             América              Europa        9   0.265   
1               China               Japão        8   0.381   
2          Inglaterra      Rio de Janeiro        8   0.145   
3              Brasil      Rio de Janeiro        8   0.151   
4              França          Inglaterra        8   0.195   
5          Inglaterra             Londres        8   0.205   
6               Paulo               Pedro        6   0.667   
7              Europa              França        6   0.136   
8              Europa  Jornal do Comércio        5   0.119   
9             Espanha          Inglaterra        5   0.143   

  top 10 por Jaccard (contagem ≥ 3)                                    
                                  u                v contagem jaccard  
0                       Teresópolis        Vassouras        3   1.000  
1                          Martinho           Sancho        3   0.750  
2                             Isaac             Jacó        4   0.667  
3                             Paulo            Pedro        6   0.667  
4                            Fausto        Margarida        4   0.571  
5                             Byron             Hugo        4   0.500  
6                              Cuba          Dongola        3   0.500  
7                           Aquiles           Heitor        3   0.429  
8                             Sarah  Sarah Bernhardt        3   0.429  
9                      José Basílio          Uruguai        3   0.429

As duas listas quase não se cruzam. A contagem devolve os **hubs** (Rio de Janeiro com todo mundo). O Jaccard devolve os **pares exclusivos**, que muitas vezes são a informação mais específica do corpus (e, de vez em quando, um erro de resolução). Nenhuma das duas é "a certa". Quem escolhe é a pergunta.

## 6.5 · O esqueleto, agora com tipos

Como no romance, subimos o limiar. Com peso ≥ 3 sobram cerca de cem nós e a rede se parte em um componente grande e vários pares. Os tipos, que na Seção 5 eram todos `PER`, agora entram como cor.

> 🧠 **Gestalt em ação · Similaridade + Proximidade.** Cor é tipo, tamanho é número de crônicas, distância é força de co-ocorrência. Note como os lugares (laranja) formam a espinha dorsal do componente grande, e como as pessoas (azul) se penduram nele. Os pares soltos na periferia são os "exclusivos" do Jaccard.

In [37]:
def subgrafo_por_limiar(g, w):
    h = nx.Graph()
    h.add_edges_from((u, v, d) for u, v, d in g.edges(data=True) if d["weight"] >= w)
    for n in h.nodes():
        h.nodes[n].update(g.nodes[n])
    return h

E_cr = subgrafo_por_limiar(G_cr, 3)
comps = sorted(nx.connected_components(E_cr), key=len, reverse=True)
print(E_cr, "·", len(comps), "componentes · tamanhos:", [len(c) for c in comps])
print("componentes pequenos:", [" · ".join(sorted(c)) for c in comps[1:]])

pos_e = nx.spring_layout(E_cr, seed=42, weight="weight", k=0.55)
nos_e = list(E_cr.nodes())
fig = figura_rede(tracos_arestas_ponderadas(E_cr, pos_e, cor="#8f9bb3", opacidade=0.5)
                  + [traco_nos(E_cr, pos_e, [TIPO_COR[E_cr.nodes[n]["tipo"]] for n in nos_e],
                               tamanho_por([E_cr.nodes[n]["unidades"] for n in nos_e], 7, 34),
                               [hover_no(E_cr, n) for n in nos_e], rotulos=True, posicao_texto="top center", tamanho_fonte=9)],
                  "A Semana · esqueleto (peso ≥ 3) · cor = tipo · tamanho = crônicas", altura=720)
legenda_tipos(fig)
fig.show()

Graph with 108 nodes and 200 edges · 9 componentes · tamanhos: [87, 5, 4, 2, 2, 2, 2, 2, 2]
componentes pequenos: ['Jerusalém · Martinho · Paulo · Pedro · Sancho', 'Alá · Byron · Hugo · Turquia', 'Babilônia · Verdi', 'Botafogo · Laranjeiras', 'Cleópatra · Egito', 'Fausto · Margarida', 'Isaac · Jacó', 'Sarah · Sarah Bernhardt']


## 6.6 · Que tipo anda com que tipo

Com tipos nos nós, a rede vira uma **rede multi-tipo** e permite perguntas que o romance não permitia. A tabela cruza os tipos nas duas pontas de cada aresta da projeção completa. E a segunda tabela responde, para as pessoas mais citadas, qual lugar mais aparece ao lado delas.

In [38]:
cruz = collections.Counter(tuple(sorted((tipo_cr[u], tipo_cr[v]))) for u, v in G_cr.edges())
tab_cruz = pd.DataFrame(0, index=list(TIPO_COR), columns=list(TIPO_COR))
for (a, b), n in cruz.items():
    tab_cruz.loc[a, b] += n
    if a != b:
        tab_cruz.loc[b, a] += n
display(tab_cruz.rename(index=TIPO_NOME, columns=TIPO_NOME))

pessoas_top = nos_cr[nos_cr.tipo == "PER"].head(10).index
linhas = []
for p in pessoas_top:
    lugares = sorted(((G_cr[p][v]["weight"], v) for v in G_cr[p] if tipo_cr[v] == "LOC"), reverse=True)[:3]
    linhas.append({"pessoa": p, "crônicas": G_cr.nodes[p]["unidades"], "lugares que mais a acompanham": ", ".join(f"{v} ({w})" for w, v in lugares)})
pd.DataFrame(linhas).set_index("pessoa")

,pessoa,lugar,organização,outro
pessoa,1124,2386,147,284
lugar,2386,1531,231,313
organização,147,231,6,29
outro,284,313,29,21


,crônicas,lugares que mais a acompanham
pessoa,,
Jesus,18,"Rio de Janeiro (4), São Mateus (3), Sergipe (2)"
César,15,"Rua do Ouvidor (4), Rio de Janeiro (3), Europa..."
Homero,15,"Londres (3), Rua do Ouvidor (2), Paraguai (2)"
Shakespeare,15,"Espanha (3), Washington (2), Rio de Janeiro (2)"
José Rodrigues,14,"Rua do Ouvidor (2), Marte (2), Júpiter (2)"
Camões,14,"Rua do Ouvidor (3), Paris (2), Washington (1)"
S. Paulo,12,"Brasil (4), Pará (3), Rio de Janeiro (2)"
Eva,11,"Bahia (3), Rio de Janeiro (2), Paris (2)"
Gonçalves Dias,10,"Rua do Ouvidor (4), Maranhão (3), Inglaterra (3)"


## 6.7 · A rede no tempo

Cada crônica tem uma data, então podemos perguntar o que cada ano "coloca junto". A rede completa é a soma das redes anuais. Vamos desenhar as de 1892 a 1897 (1900 tem só dois textos) sobre **as mesmas posições** e deixar o ano no controle deslizante.

> 🧠 **Gestalt em ação · Destino comum + Figura e fundo.** Os nós que não aparecem no ano ficam cinza e pequenos; os que aparecem acendem com a cor do tipo. Mova o controle e observe grupos inteiros acendendo e apagando. 1894 e 1895 acendem China e Japão (a guerra sino-japonesa). 1896 acende Antônio Conselheiro (Canudos), ligado a Bahia e Sergipe, escondido no meio do aglomerado central (passe o mouse, ou procure-o no mapa de calor logo abaixo). O título de cada quadro lista as cinco entidades mais fortes do ano, um barômetro do noticiário. Só entram pares que dividem ao menos duas crônicas do ano, para que o quadro não vire uma bola de pelos.

In [39]:
ano_de = cronicas.set_index(cronicas.id.map(lambda i: f"c{i}")).ano.to_dict()

def rede_do_ano(ano, min_peso=2):
    docs_ano = [d for d in docs_ids if ano_de[d] == ano]
    sub = B_cr.subgraph(docs_ano + entidades)
    g = bipartite.weighted_projected_graph(sub, entidades)
    h = nx.Graph((u, v, d) for u, v, d in g.edges(data=True) if d["weight"] >= min_peso)
    for n in h.nodes():
        h.nodes[n]["unidades"] = sub.degree(n)
        h.nodes[n]["tipo"] = tipo_cr[n]
    return h

anos = [a for a in sorted(cronicas.ano.unique()) if (cronicas.ano == a).sum() >= 5]
redes_ano = {a: rede_do_ano(a) for a in anos}
G_uniao = nx.compose_all(list(redes_ano.values()))            # união das redes anuais, para um layout fixo e arejado
pos_ano = nx.spring_layout(G_uniao, seed=7, weight="weight", k=0.9)
nos_canvas = sorted(G_uniao.nodes())

cortes_ano = np.array([2.0, 3.0, 5.0])          # classes de largura fixas para todos os anos
frames, passos = [], []
for a in anos:
    g = redes_ano[a]
    ativos = set(g.nodes())
    forca = dict(g.degree(weight="weight"))
    top5 = ", ".join(n for n, _ in sorted(forca.items(), key=lambda kv: -kv[1])[:5])
    cores = [TIPO_COR[tipo_cr[n]] if n in ativos else CINZA_NO for n in nos_canvas]
    tam = [10 + 3 * math.sqrt(g.nodes[n]["unidades"]) if n in ativos else 5 for n in nos_canvas]
    opac = [1.0 if n in ativos else 0.3 for n in nos_canvas]
    hover = [f"<b>{n}</b><br>{a}: {g.nodes[n]['unidades']} crônicas · força {forca[n]}" if n in ativos else f"{n}<br>ausente em {a}" for n in nos_canvas]
    dados = tracos_arestas_ponderadas(g, pos_ano, cor="#8f9bb3", opacidade=0.55, cortes=cortes_ano)   # sempre 4 traces
    dados.append(go.Scatter(x=[pos_ano[n][0] for n in nos_canvas], y=[pos_ano[n][1] for n in nos_canvas], mode="markers+text",
                            text=[n if n in ativos and g.nodes[n]["unidades"] >= 2 else "" for n in nos_canvas], textposition="top center",
                            textfont=dict(size=9, color="#0b0b0b"), hovertext=hover, hoverinfo="text",
                            marker=dict(color=cores, size=tam, opacity=opac, line=dict(color="white", width=1)), showlegend=False))
    titulo = f"A Semana em {a} · {(cronicas.ano == a).sum()} crônicas · {g.number_of_edges()} pares (peso ≥ 2)<br><sup>mais fortes: {top5}</sup>"
    frames.append(go.Frame(name=str(a), data=dados, layout=go.Layout(title=dict(text=titulo))))
    passos.append(dict(method="animate", label=str(a), args=[[str(a)], dict(mode="immediate", frame=dict(duration=0, redraw=True), transition=dict(duration=200))]))

fig = figura_rede(frames[0].data, "", altura=720)
fig.frames = frames
fig.update_layout(title=frames[0].layout.title, margin=dict(l=10, r=10, t=70, b=90),
                  sliders=[dict(active=0, steps=passos, x=0.02, len=0.96, y=0.0, currentvalue=dict(prefix="ano: ", font=dict(size=13)))],
                  updatemenus=[dict(type="buttons", x=1.0, y=1.0, xanchor="right", yanchor="bottom", direction="left",
                                    buttons=[dict(label="▶ tocar", method="animate", args=[None, dict(frame=dict(duration=1500, redraw=True), fromcurrent=True)]),
                                             dict(label="⏸ pausar", method="animate", args=[[None], dict(mode="immediate", frame=dict(duration=0, redraw=False))])])])
legenda_tipos(fig)
fig.update_layout(legend=dict(orientation="v", x=0.99, y=0.03, xanchor="right", yanchor="bottom", bgcolor="rgba(252,252,251,0.85)"))
fig.show()

> 🧠 **Gestalt em ação · Similaridade + Continuidade.** O mesmo dado como mapa de calor. Linhas são as entidades que mais mudam de ano para ano, colunas são os anos, a cor é o número de crônicas do ano em que a entidade aparece, em uma única matiz azul. Uma linha com uma mancha escura isolada é um assunto "de um ano". Uma linha uniforme é um assunto permanente (Rio de Janeiro, Rua do Ouvidor).

In [40]:
pres = presencas.assign(ano=presencas.cronica.map(cronicas.set_index("id").ano))
calor = pres.groupby(["no", "ano"]).cronica.nunique().unstack(fill_value=0)[anos]
calor = calor[calor.sum(axis=1) >= 5]
variacao = (calor.div(calor.sum(axis=1), axis=0).max(axis=1)).sort_values(ascending=False)   # concentração em um único ano
sel = list(variacao.head(18).index) + [n for n in ["Rio de Janeiro", "Rua do Ouvidor", "Europa"] if n in calor.index]
calor_sel = calor.loc[sel]
escala_azul = [[0, "#ffffff"]] + [[(k + 1) / len(AZUIS), c] for k, c in enumerate(AZUIS)]
fig = go.Figure(go.Heatmap(z=calor_sel.values, x=[str(a) for a in anos], y=calor_sel.index,
                           colorscale=escala_azul, hovertemplate="%{y} · %{x}<br>%{z} crônicas<extra></extra>",
                           colorbar=dict(title="crônicas", thickness=12), xgap=2, ygap=2))
fig.update_layout(title="Assuntos de um ano só (em cima) e assuntos permanentes (embaixo)", height=620, plot_bgcolor=SUPERFICIE,
                  paper_bgcolor=SUPERFICIE, margin=dict(l=10, r=10, t=50, b=10), yaxis=dict(autorange="reversed"))
fig.show()

> ✅ **O que fica.** Quando a unidade é o documento, o tamanho do documento vira estrutura falsa, o artefato do clique, e a projeção bruta tem densidade, k-core e clustering inflados. Pesos por contagem revelam hubs; Jaccard revela pares exclusivos e denuncia aliases não resolvidos. Tipos transformam a rede em multi-tipo e abrem perguntas novas (que tipo anda com que tipo). Datas transformam a foto em filme, e o filme é um barômetro do que o texto estava "colocando junto" em cada época.

# 7 · Desafios

Seis desafios, no formato de sempre. Os três primeiros vêm **resolvidos**, para estudo. Leia o enunciado, tente antes de abrir a solução, depois compare. Os três últimos **não têm solução no notebook**. Têm critérios de aceitação e pistas, e alguns tocam em perguntas de pesquisa em aberto. Entregue-os como um notebook separado.

| # | Desafio | Estado | Dificuldade |
|---|---|---|---|
| 1 | $B^{\top}B$ com matrizes esparsas | resolvido | ★☆☆ |
| 2 | Janelas de $w$ sentenças, a curva inteira | resolvido | ★★☆ |
| 3 | Troque a ferramenta, mantenha a rede | resolvido | ★★☆ |
| 4 | A rede no tempo do romance | aberto | ★★★ |
| 5 | Outro romance, outro narrador | aberto | ★★★ |
| 6 | Wikipédia ao vivo, e os links como resolução de identidade | aberto | ★★★★ |

## Desafio 1 · $B^{\top}B$ com matrizes esparsas (resolvido)

**Enunciado.** (a) Escreva `matriz_incidencia(df, col_unidade)` que devolva a matriz $B$ (unidades × nós) como `scipy.sparse.csr_matrix`, mais as listas de rótulos das linhas e das colunas. (b) Calcule $C = B^{\top}B$ e prove com `assert` que os pesos coincidem com `g_cap` e que a diagonal coincide com o atributo `unidades`. (c) Meça o tempo da construção por pares (`rede_coocorrencia`) e por matriz nas crônicas, e explique a diferença a partir da Seção 2.3.

<details><summary><b>Pista</b></summary>
Uma matriz esparsa em formato COO é construída a partir de três listas (valores, linhas, colunas). Cada linha da tabela de presenças vira um 1. Não esqueça de remover duplicatas (a mesma entidade duas vezes na mesma unidade deve virar um único 1).
</details>

In [ ]:
# ---- Solução do Desafio 1 ----
from scipy import sparse

def matriz_incidencia(df, col_unidade, col_no="no"):
    d = df.dropna(subset=[col_no])[[col_unidade, col_no]].drop_duplicates()
    unidades = sorted(d[col_unidade].unique(), key=str)
    nos = sorted(d[col_no].unique())
    iu = {u: k for k, u in enumerate(unidades)}
    ino = {n: k for k, n in enumerate(nos)}
    B = sparse.csr_matrix((np.ones(len(d)), (d[col_unidade].map(iu).values, d[col_no].map(ino).values)),
                          shape=(len(unidades), len(nos)))
    return B, unidades, nos

B_dc, U_dc, N_dc = matriz_incidencia(mencoes, "capitulo")
C_dc = (B_dc.T @ B_dc).toarray().astype(int)
ino = {n: k for k, n in enumerate(N_dc)}
assert all(C_dc[ino[u], ino[v]] == d["weight"] for u, v, d in g_cap.edges(data=True))
assert all(C_dc[ino[n], ino[n]] == g_cap.nodes[n]["unidades"] for n in g_cap.nodes())
assert (np.triu(C_dc, 1) > 0).sum() == g_cap.number_of_edges()
print(f"✅ B tem forma {B_dc.shape}, C = BᵀB coincide com g_cap ({g_cap.number_of_edges()} arestas) e com a diagonal")

# (c) tempo nas crônicas: pares vs matriz
pres_cr = mencoes_cr.dropna(subset=["no"])
t0 = time.perf_counter(); g_pares = rede_coocorrencia(pres_cr, "cronica"); t_pares = time.perf_counter() - t0
t0 = time.perf_counter(); B_c, U_c, N_c = matriz_incidencia(pres_cr, "cronica"); C_c = B_c.T @ B_c; t_mat = time.perf_counter() - t0
print(f"crônicas: pares {t_pares * 1000:.0f} ms · matriz esparsa {t_mat * 1000:.0f} ms · nnz(C) = {C_c.nnz} (inclui a diagonal)")

✅ B tem forma (129, 17), C = BᵀB coincide com g_cap (93 arestas) e com a diagonal
crônicas: pares 13 ms · matriz esparsa 4 ms · nnz(C) = 12448 (inclui a diagonal)


**(c) Discussão.** Os dois métodos fazem o mesmo trabalho, $O\!\left(\sum_u k_u^2\right)$, porque $B^{\top}B$ esparso também percorre, para cada unidade, todos os pares de itens presentes. A diferença de tempo é de **implementação** (laço em Python contra laço em C), não de complexidade. A vantagem prática da matriz aparece quando se quer normalizar em bloco (Jaccard, cosseno, PMI são operações sobre $C$ e sua diagonal) ou quando as unidades são milhões e cabem em memória só como esparsas. A vantagem dos pares é a transparência, e é por isso que foi o método da aula.

## Desafio 2 · Janelas de $w$ sentenças, a curva inteira (resolvido)

**Enunciado.** Na Seção 5.8 comparamos três janelas. Generalize. Para $w \in \{1, 2, 3, 5, 10, 20, 50, 100\}$, defina a unidade como o **bloco** de $w$ sentenças consecutivas dentro do capítulo (bloco $= \lfloor \text{índice da sentença} / w \rfloor$), construa a rede e registre número de arestas, densidade, clustering médio, número de componentes e aresta mais pesada. Plote as curvas em função de $w$ (eixo x logarítmico) e responda, a partir de que $w$ a rede coincide com a rede por capítulo, e por quê.

> 🧠 **Gestalt em ação · Continuidade.** Quatro curvas, uma por medida, contra $w$. A forma de cada curva é o efeito da janela. Onde a curva fica plana, aumentar a janela não muda mais nada.

In [ ]:
# ---- Solução do Desafio 2 ----
def rede_por_blocos(df, w):
    d = df.dropna(subset=["no"]).copy()
    d["bloco"] = d.capitulo.astype(str) + "." + (d.idx_sentenca // w).astype(str)
    return rede_coocorrencia(d, "bloco")

janelas = [1, 2, 3, 5, 10, 20, 50, 100]
linhas = []
for w in janelas:
    g = rede_por_blocos(mencoes, w)
    top_w, top_u, top_v = max((d["weight"], u, v) for u, v, d in g.edges(data=True))
    linhas.append({"w": w, "arestas": g.number_of_edges(), "densidade": nx.density(g), "clustering": nx.average_clustering(g),
                   "componentes": nx.number_connected_components(g), "aresta mais pesada": f"{top_u} · {top_v} ({top_w})"})
curva = pd.DataFrame(linhas).set_index("w")
display(curva.round(3))

fig = make_subplots(rows=1, cols=4, subplot_titles=("arestas", "densidade", "clustering médio", "componentes"))
for k, col in enumerate(["arestas", "densidade", "clustering", "componentes"], start=1):
    fig.add_trace(go.Scatter(x=curva.index, y=curva[col], mode="lines+markers", line=dict(color=AZUIS[5], width=2.5),
                             marker=dict(size=9, color=AZUIS[5], line=dict(color="white", width=1)), showlegend=False,
                             hovertemplate="w = %{x}<br>" + col + " = %{y:.3f}<extra></extra>"), row=1, col=k)
    fig.add_hline(y={"arestas": g_cap.number_of_edges(), "densidade": nx.density(g_cap), "clustering": nx.average_clustering(g_cap),
                     "componentes": nx.number_connected_components(g_cap)}[col], line=dict(color=DESTAQUE, dash="dot"), row=1, col=k)
fig.update_xaxes(type="log", title_text="w (sentenças por bloco)", gridcolor=GRADE, tickvals=janelas, ticktext=[str(w) for w in janelas])
fig.update_yaxes(gridcolor=GRADE)
fig.update_layout(height=380, plot_bgcolor=SUPERFICIE, paper_bgcolor=SUPERFICIE, margin=dict(l=10, r=10, t=80, b=10),
                  title=dict(text="Efeito da janela · linha vermelha = valor na rede por capítulo", x=0.02, y=0.97))
fig.show()
sent_por_cap = mencoes.groupby("capitulo").idx_sentenca.max() + 1
print(f"sentenças por capítulo: mediana {sent_por_cap.median():.0f}, máximo {sent_por_cap.max()}")

,arestas,densidade,clustering,componentes,aresta mais pesada
w,,,,,
1,45,0.331,0.579,2,José Dias · Tio Cosme (13)
2,52,0.382,0.679,2,Capitu · Escobar (20)
3,54,0.397,0.677,2,Capitu · Escobar (28)
5,67,0.493,0.715,1,Capitu · Escobar (32)
10,78,0.574,0.795,1,Capitu · José Dias (42)
20,89,0.654,0.836,1,Capitu · José Dias (45)
50,92,0.676,0.851,1,Capitu · José Dias (45)
100,93,0.684,0.855,1,Capitu · José Dias (41)


sentenças por capítulo: mediana 23, máximo 116


**Discussão.** A curva converge para a rede por capítulo quando $w$ se aproxima do tamanho do capítulo mais longo (116 sentenças), porque então quase todo capítulo cabe em um único bloco e as duas definições de unidade coincidem (com $w = 100$ só um capítulo é dividido em dois blocos, e essa divisão não separa nenhum par). Antes disso, cada medida conta uma história. As arestas e a densidade crescem monotonicamente (juntar blocos só pode acrescentar pares). O clustering tende a crescer porque blocos maiores geram cliques maiores. Os componentes caem porque personagens que nunca dividem uma frase passam a dividir um bloco. A aresta mais pesada muda de dono ao longo do caminho, de José Dias · Tio Cosme para Capitu · Escobar e depois para Capitu · José Dias, o que mostra que "quem está mais junto de quem" é uma função da janela, não uma propriedade do texto.

## Desafio 3 · Troque a ferramenta, mantenha a rede (resolvido)

**Enunciado.** A tese da aula é que o NER é substituível. Prove. (a) Escreva `extrair_por_lista(texto, aliases)` que encontre, **sem NER**, as ocorrências das formas do dicionário `ALIASES` por expressão regular (a ferramenta mais antiga do mundo, uma lista de nomes). (b) Construa a rede por capítulo com essa ferramenta. (c) Compare com `g_cap` em quatro números, nós em comum, Jaccard entre os conjuntos de arestas, correlação de Spearman entre os pesos das arestas comuns, e as cinco arestas mais pesadas de cada uma. (d) Escreva uma frase sobre o que a troca de ferramenta muda e o que não muda.

In [ ]:
# ---- Solução do Desafio 3 ----
def extrair_por_lista(texto, aliases=ALIASES):
    linhas = []
    for canonico, formas in aliases.items():
        padrao = re.compile(r"(?<![\w])(" + "|".join(re.escape(f) for f in sorted(formas, key=len, reverse=True)) + r")(?![\w])")
        for m in padrao.finditer(texto):
            linhas.append({"mencao": m.group(1), "no": canonico, "inicio": m.start()})
    return pd.DataFrame(linhas)

men_lista = pd.concat([extrair_por_lista(c).assign(capitulo=k) for k, c in enumerate(capitulos, start=1)], ignore_index=True)
g_lista = rede_coocorrencia(men_lista, "capitulo")

A_ner = {frozenset(e) for e in g_cap.edges()}
A_lista = {frozenset(e) for e in g_lista.edges()}
comuns = A_ner & A_lista
pesos = pd.DataFrame([{"ner": g_cap.edges[tuple(e)]["weight"], "lista": g_lista.edges[tuple(e)]["weight"]} for e in comuns])
print("nós NER:", g_cap.number_of_nodes(), "| nós lista:", g_lista.number_of_nodes(), "| em comum:", len(set(g_cap) & set(g_lista)),
      "| só no NER:", sorted(set(g_cap) - set(g_lista)))
print(f"arestas NER {len(A_ner)} · lista {len(A_lista)} · Jaccard entre os conjuntos de arestas = {len(comuns) / len(A_ner | A_lista):.2f}")
print(f"Spearman entre os pesos das {len(comuns)} arestas comuns = {pesos.corr('spearman').iloc[0, 1]:.3f}")
top = lambda g: [f"{u} · {v} ({d['weight']})" for u, v, d in sorted(g.edges(data=True), key=lambda e: -e[2]['weight'])[:5]]
pd.DataFrame({"NER + dicionário": top(g_cap), "só lista de nomes": top(g_lista)})

nós NER: 17 | nós lista: 16 | em comum: 16 | só no NER: ['Desdêmona']
arestas NER 93 · lista 89 · Jaccard entre os conjuntos de arestas = 0.96
Spearman entre os pesos das 89 arestas comuns = 0.982


,NER + dicionário,só lista de nomes
0,Capitu · José Dias (40),Capitu · José Dias (40)
1,Capitu · Escobar (32),Capitu · Escobar (32)
2,Bentinho · Capitu (29),Bentinho · Capitu (30)
3,Bentinho · José Dias (26),Bentinho · José Dias (28)
4,Capitu · D. Glória (24),José Dias · Prima Justina (23)


**(d)** Para o elenco principal, a rede **não depende da ferramenta**. Os mesmos pares, quase os mesmos pesos (a lista acha mais menções, porque não perde nomes que o modelo deixa passar, e o NER acha o que não estava na lista, como Desdêmona). O que a ferramenta muda é a **periferia** e o **custo de entrada**. Uma lista de nomes exige saber os nomes de antemão, o NER descobre nomes em um corpus desconhecido (foi assim nas crônicas), ao preço do ruído. A estrutura da rede está no texto; a ferramenta só decide quanto dela enxergamos.

## Desafio 4 · A rede no tempo do romance (aberto)

A rede da Seção 5 soma os 148 capítulos em uma foto. Mas o romance é um filme, e a Seção 5.9 mostrou que o capítulo de entrada de cada personagem já carrega estrutura.

**Tarefa.**

1. Construa a rede **acumulada** até o capítulo $t$, para $t = 1, \dots, 148$, e registre número de nós, arestas e densidade ao longo de $t$. Plote as três séries.
2. Faça uma **animação** (como as das Seções 5.7 e 6.7) com posições fixas, em que os personagens acendem no capítulo em que entram e as arestas engrossam à medida que os pesos acumulam.
3. Defina uma medida de **novidade** do capítulo $t$ (por exemplo, o número de arestas que aparecem pela primeira vez em $t$) e encontre os cinco capítulos que mais mudam a rede. Leia os títulos desses capítulos (a lista `titulos`) e diga se fazem sentido como pontos de virada.
4. Construa também a rede por **janela deslizante** de 10 capítulos e compare com a acumulada. Qual das duas mostra melhor a entrada de Escobar? E a de Ezequiel?

**Critérios de aceitação.** Animação com controle deslizante e botão de tocar; a medida de novidade deve ser definida em uma frase e justificada; a comparação do item 4 deve vir com uma figura de posições fixas.

**Pista.** `mencoes.groupby("no").capitulo.min()` dá o capítulo de entrada. Para a rede acumulada, filtre `mencoes[mencoes.capitulo <= t]` e reaproveite `rede_coocorrencia`.

**Por que é aberto.** Não existe definição canônica de "rede de personagens dinâmica". Há propostas por janela, por acumulação, por decaimento exponencial, e cada uma responde a uma pergunta literária diferente. Uma boa resposta escolhe uma e defende.

In [ ]:
# ---- Desafio 4: seu código aqui ----

## Desafio 5 · Outro romance, outro narrador (aberto)

*Dom Casmurro* é narrado em primeira pessoa, e vimos o efeito, Bentinho é um nó fantasma. *Quincas Borba* (1891, `romance/marm07.txt`) é narrado em terceira pessoa. *Memórias Póstumas de Brás Cubas* (1881, `romance/marm05.txt`) é narrado em primeira pessoa por um defunto.

**Tarefa.**

1. Rode o pipeline inteiro (fatiar, extrair, resolver, ligar) em um dos dois romances. Você terá de escrever o dicionário de aliases, o que exige olhar a lista suja com atenção. Registre quantas formas você precisou fundir.
2. Compare as três "assinaturas" (a rede de *Dom Casmurro* e a do romance escolhido, mais uma rede aleatória com o mesmo número de nós e arestas) em densidade, distribuição de força, clustering e esqueleto por limiar.
3. Responda com números, o narrador em terceira pessoa de *Quincas Borba* aparece na rede na proporção da sua presença? Como você mediria "presença" sem usar o nome?
4. (Ousado) Franco Moretti propôs que a estrutura da rede de personagens é uma **assinatura do autor**. Com dois romances do mesmo autor, o que você diria? O que faltaria para responder de verdade?

**Critérios de aceitação.** Dicionário de aliases explícito e comentado; a comparação do item 2 com ao menos uma figura de posições fixas ou pequenos múltiplos; o item 3 com uma proposta concreta de medida (mesmo que imperfeita).

**Por que é aberto.** A extração de redes de personagens de ficção é um campo ativo (veja Labatut e Bost, 2019, nas referências) e a questão do narrador não tem solução puramente algorítmica.

In [ ]:
# ---- Desafio 5: seu código aqui ----

## Desafio 6 · Wikipédia ao vivo, e os links como resolução de identidade (aberto)

Tudo nesta aula foi texto do século XIX. Agora um corpus vivo. A API da Wikipédia devolve o texto puro de qualquer artigo, e os artigos têm uma coisa que nenhum romance tem, **links**. Cada link é uma menção já resolvida por humanos (a âncora aponta para uma página única, "Getúlio Vargas" e "Vargas" apontam para o mesmo lugar). É uma segunda ferramenta de extração, alternativa ao NER.

**Tarefa.**

1. Escolha um artigo longo da Wikipédia em português (por exemplo, `Segunda Guerra Mundial`, `Guerra Fria` ou `História do Brasil`). Baixe o texto puro com a célula abaixo e fatie em parágrafos.
2. Construa a rede de co-ocorrência por parágrafo com **NER** (a peneira automática da Seção 6, todos os tipos).
3. Construa a mesma rede usando os **links** do artigo como itens (a API com `prop=links` ou o HTML do artigo). Compare as duas redes como no Desafio 3.
4. Desenhe a rede final com cor por tipo e discuta o que os tipos revelam sobre o assunto (por exemplo, países e líderes em uma guerra).
5. (Ousado) Estenda para os artigos vizinhos por *snowball* (como na aula de Small World) e construa a rede de entidades da vizinhança inteira. O artefato do clique volta com força. Trate-o.

**Critérios de aceitação.** As duas redes (NER e links) com os quatro números do Desafio 3; uma figura com tipos; um parágrafo sobre onde os links acertam e o NER erra, e vice-versa.

**Por que é aberto.** A "resolução de entidades" (decidir que duas menções são a mesma coisa) é um problema de pesquisa por si só. Os links da Wikipédia são uma solução parcial e gratuita, e é instrutivo medir o quanto ela vale.

In [ ]:
# ---- Desafio 6: ponto de partida ----
# A célula abaixo depende de acesso à Internet (pt.wikipedia.org).
# import requests
# def texto_wikipedia(titulo, lingua="pt"):
#     r = requests.get(f"https://{lingua}.wikipedia.org/w/api.php",
#                      params={"action": "query", "prop": "extracts", "explaintext": 1, "redirects": 1, "format": "json", "titles": titulo},
#                      headers={"User-Agent": "aula-grafos-ufrn/1.0 (contato: seu-email@ufrn.br)"}, timeout=30)
#     r.raise_for_status()
#     pagina = next(iter(r.json()["query"]["pages"].values()))
#     return pagina.get("title"), pagina.get("extract", "")
# titulo, texto = texto_wikipedia("Segunda Guerra Mundial")
# paragrafos = [p for p in texto.split("\n") if len(p.strip()) > 80]
# print(titulo, len(texto), "caracteres,", len(paragrafos), "parágrafos")

# 8 · Quiz relâmpago, checklist e referências

Responda de cabeça e só depois abra a resposta.

<details><summary><b>1. Dois personagens aparecem no mesmo capítulo mas nunca na mesma sentença. Em qual das três redes da Seção 5.8 eles são vizinhos?</b></summary>
Na rede por capítulo, sim. Na rede por parágrafo, só se dividirem algum parágrafo. Na rede por sentença, não. A vizinhança depende da janela, não só do texto.
</details>

<details><summary><b>2. Uma crônica cita 20 entidades. Quantas arestas ela cria sozinha na projeção? E uma com 40?</b></summary>
20·19/2 = 190 e 40·39/2 = 780. Dobrar o tamanho da unidade quadruplica as arestas. É o artefato do clique, e é por isso que poucas unidades grandes dominam a projeção.
</details>

<details><summary><b>3. A degenerescência da projeção de A Semana é alta. Isso significa que há um grupo coeso de entidades?</b></summary>
Não necessariamente. Cada documento vira um clique, e cliques sobrepostos formam um k-core alto por construção. Em projeções, o k-core mede, em boa parte, tamanho de documento. Para achar coesão real é preciso olhar pesos normalizados.
</details>

<details><summary><b>4. Jaccard igual a 1 entre duas entidades significa o quê? E Jaccard alto entre "Sarah" e "Sarah Bernhardt"?</b></summary>
Que as duas aparecem exatamente nas mesmas unidades e em nenhuma outra. Entre duas formas parecidas, é um sinal forte de que são a mesma entidade, um alias não resolvido.
</details>

<details><summary><b>5. Por que Bentinho tem força menor que Capitu, se ele está em todos os capítulos?</b></summary>
Porque a ferramenta conta nomes, e um narrador em primeira pessoa raramente se nomeia. A rede é a das pessoas que o narrador nomeia; ele mesmo é um nó fantasma.
</details>

<details><summary><b>6. Se o NER rotulasse Capitu como LOC em todas as menções, o que mudaria na rede de co-ocorrência?</b></summary>
Nada nas arestas nem nos pesos, porque a rede usa só os trechos e as unidades. Mudaria a cor do nó e qualquer análise por tipo. Os tipos são atributos; a topologia vem das menções.
</details>

<details><summary><b>7. Contrair os nós "Mamãe" e "D. Glória" depois de projetar dá o mesmo resultado que fundir as menções antes de projetar?</b></summary>
Não. Se as duas formas aparecem no mesmo capítulo com Capitu, a contração soma duas arestas e conta o capítulo duas vezes. Fundir na tabela conta uma vez. Projeção e contração não comutam.
</details>

### ✅ Checklist de saída

- [ ] Sei definir rede de co-ocorrência como projeção de um bipartido unidades × itens e sei calcular o peso por contagem e por Jaccard.
- [ ] Sei montar a matriz de incidência $B$ e explicar por que $B^{\top}B$ conta co-ocorrências (passeios de comprimento 2).
- [ ] Sei usar `extrair_entidades` como uma caixa preta com contrato, e sei listar três limitações do NER sem abrir a caixa.
- [ ] Sei transformar menções em nós (normalização, dicionário, peneira automática) e explicar por que a resolução tem de vir antes da projeção.
- [ ] Consigo prever o efeito da janela sobre densidade, clustering e componentes, e escolher a janela a partir da pergunta.
- [ ] Sei ler uma rede de co-ocorrência por força, intermediação ponderada e esqueleto por limiar, e sei por que o grau satura.
- [ ] Reconheço o artefato do clique em uma projeção e sei corrigi-lo com normalização.

### 📚 Referências

* Knuth, D. E. (1993). *The Stanford GraphBase: A Platform for Combinatorial Computing*. ACM Press. (A rede de *Os Miseráveis*, feita à mão.)
* Newman, M. (2018). *Networks*, 2ª ed. Oxford University Press. (Capítulo sobre redes bipartidas e projeções.)
* Coscia, M. (2021). *The Atlas for the Aspiring Network Scientist*. Capítulos sobre redes bipartidas e projeções, https://www.networkatlas.eu/.
* Elson, D. K., Dames, N. e McKeown, K. R. (2010). Extracting social networks from literary fiction. *Proceedings of ACL*.
* Labatut, V. e Bost, X. (2019). Extraction and analysis of fictional character networks: a survey. *ACM Computing Surveys*, 52(5).
* Moretti, F. (2011). Network theory, plot analysis. *New Left Review*, 68.
* Nadeau, D. e Sekine, S. (2007). A survey of named entity recognition and classification. *Lingvisticae Investigationes*, 30(1).
* Zhou, T., Ren, J., Medo, M. e Zhang, Y.-C. (2007). Bipartite network projection and personal recommendation. *Physical Review E*, 76.
* Serrano, M. Á., Boguñá, M. e Vespignani, A. (2009). Extracting the multiscale backbone of complex weighted networks. *PNAS*, 106(16). (Um esqueleto mais sofisticado que o limiar simples.)
* Honnibal, M., Montani, I., Van Landeghem, S. e Boyd, A. (2020). spaCy: Industrial-strength Natural Language Processing in Python. https://spacy.io/.
* Bird, S., Klein, E. e Loper, E. (2009). *Natural Language Processing with Python*. O'Reilly. (O corpus `machado` do NLTK.)
* Machado de Assis, J. M. (1899). *Dom Casmurro*; (1892–1897) *A Semana*. Obra Completa, Nova Aguilar, 1994, via NLTK (domínio público).